# Fase 3 — Notebook 04: Análise Fatorial e os Pesos do IVS

**Entrada:** `banco_de_dados/entrega_orientadora/Base_ELSI_70Municipios_Censo2022.db`,
tabela `setores_censitarios` — o SQLite versionado da entrega. Este notebook roda **sem**
os 2,4 GB de microdados do Censo.

**Objetivo:** estimar a estrutura latente dos componentes do IVS e definir os **pesos** do
índice. O produto são os pesos e a estrutura — **não** o IVS final, que é o Notebook 05,
depois da normalização municipal do Notebook 03.

**Referência metodológica:**
- MATOS, D. A. S.; RODRIGUES, E. C. *Análise fatorial*. Brasília: Enap, 2019. 74 p. — a
  referência principal. As páginas citadas ao longo do notebook são as da numeração
  impressa.
- FIGUEIREDO FILHO, D. B.; SILVA JÚNIOR, J. A. Visão além do alcance: uma introdução à
  análise fatorial. *Opinião Pública*, v. 16, n. 1, 2010 — a referência anterior, que a
  Enap contradiz em três pontos (rotação, técnica de extração e regra de comunalidade).
- `docs/metodologia/Analise_Fatorial_Figueiredo2010_e_o_Projeto_IVS.md` — a análise já rodada em
  24/08/2026, cujos itens 13, 14 e 15 do checklist são o escopo deste notebook.

**Onde mora a matemática:** `src/ivs_censo/fatorial.py`, testado em
`tests/test_fatorial.py`. Este notebook chama, interpreta e produz figuras — não
reimplementa nada.

**O que este notebook NÃO decide.** Quatro pontos são da orientação, e aqui se produz a
evidência e o custo de cada opção, não a escolha: o destino do indicador de lixo, o
número de fatores, a política do sigilo no analfabetismo, e se a solução oficial será
ortogonal ou oblíqua.

**Saídas:** `banco_de_dados/eda/fatorial/nb04_*.csv` e
`banco_de_dados/eda/fatorial/figuras/*.png`.

## 1. Carga e recorte

O recorte é o mesmo do Notebook 02 — `urbano = 1` e `Dados_sig = 'OK'` — e as fórmulas dos
indicadores vêm de `src/ivs_censo/indicadores.py`, nunca redefinidas aqui.

A **renda entra invertida** (−renda) para que todas as variáveis apontem no mesmo sentido:
valor maior = mais vulnerável. A inversão não muda |r|, autovalores, KMO nem
comunalidades; muda o sinal das cargas, e com ele a leitura.

A exclusão de casos é por lista (*listwise*). O custo está medido e é grande: 16.563
setores, 15,9% do recorte, quase todos pelo sigilo do analfabetismo — e a §6.2.6 do
`GUIA_DO_PROJETO.md` documenta que esse sigilo **não é aleatório**, incide nos setores de
melhor situação. O livro da Enap não trata de dados faltantes em nenhuma das 74 páginas;
a lacuna fica declarada no bloco 10.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os                          # caminhos e criação de pastas
import sys                         # para registrar src/ no caminho de importação
import sqlite3                     # leitura do banco da entrega (o .db, não o CSV)
import numpy as np                 # álgebra: matrizes, autovalores, álgebra linear
import pandas as pd                # DataFrames: as tabelas e os CSVs de saída
import matplotlib.pyplot as plt    # figuras
from pathlib import Path           # caminhos como objetos, e não como texto

pd.set_option('display.max_columns', 50)   # ao imprimir, mostra até 50 colunas sem truncar
pd.set_option('display.width', 200)        # largura da saída: evita quebrar tabela no meio


def _find_project_root():
    """Detecta a raiz do projeto (independente de onde o Jupyter inicia o kernel)."""
    cwd = Path.cwd().resolve()              # de onde o kernel foi iniciado, em caminho absoluto
    for d in [cwd, *cwd.parents]:           # sobe a árvore: a pasta atual e todas as acima
        # a raiz é a única que tem os três ao mesmo tempo — é a assinatura do projeto
        if (d / 'requirements.txt').is_file() and (d / 'dados').is_dir() and (d / 'docs').is_dir():
            return d
    raise RuntimeError(f'Raiz do projeto não encontrada a partir de: {cwd}')  # falha explícita

ROOT = _find_project_root()                                    # a raiz, achada uma vez só
# O NB04 lê o SQLITE da entrega, não os 2,4 GB de microdados do Censo: tudo de que ele
# precisa já está calculado ali, e assim o notebook roda em qualquer máquina.
CAMINHO_DB  = ROOT / 'banco_de_dados' / 'entrega_orientadora' / 'Base_ELSI_70Municipios_Censo2022.db'
CAMINHO_FAT = ROOT / 'banco_de_dados' / 'eda' / 'fatorial'     # onde os CSVs de saída vão
CAMINHO_FIG = CAMINHO_FAT / 'figuras'                          # e onde as figuras vão
CAMINHO_FIG.mkdir(parents=True, exist_ok=True)                 # cria a pasta se não existir

# A matemática vive no módulo, testada. Aqui só se chama.
sys.path.insert(0, str(ROOT / 'src'))       # põe src/ no caminho de importação do Python
from ivs_censo.fatorial import (IVS7, ROTULOS, acp, bartlett, bootstrap_cargas,   # noqa: E402
                                comunalidades_obliquas, escores_regressao,
                                fatoracao_eixo_principal, horn, kmo,
                                matriz_correlacao, postos, rotacao_promax, smc, varimax)
# IVS7 = os nomes das 7 colunas · ROTULOS = o nome de leitura de cada uma
# acp = componentes principais · bartlett/kmo/smc = testes de adequabilidade
# horn = análise paralela · varimax/rotacao_promax = rotações · escores_regressao = B = R⁻¹A

# Paleta do projeto — a mesma dos decks e dos relatórios.
TINTA, PETROL, CLAY, CINZA = '#1A1A1A', '#1F4E4A', '#A83A2C', '#666666'   # tinta/verde/vermelho/cinza
SURF = '#FCFCFB'                           # o fundo das figuras: papel, não branco puro

# NOTA SOBRE COR: petrol e clay, usados como par categórico, ficam a ΔE 6,1 em
# protanopia — abaixo do piso de 8. A paleta é a identidade do projeto e não muda aqui;
# as figuras é que foram desenhadas para não depender de cor sozinha: rótulo direto em
# cada marca, traço cheio contra tracejado, e valor impresso em cada célula do mapa de
# cargas. Cor é reforço, nunca o único canal.

# Eixos em português: o separador decimal é vírgula, como no resto dos documentos.
# FuncFormatter recebe o valor e a posição; devolvemos o texto do rótulo do eixo.
VIRGULA = plt.matplotlib.ticker.FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ','))


# Rampa divergente da paleta do projeto: usada pela matriz de correlação e
# pelo mapa de cargas, que são as duas figuras de grandeza COM SINAL.
def rampa_divergente(v):
    """Interpola cinza-claro -> clay (positivo) ou cinza-claro -> petrol (negativo)."""
    neutro = np.array([0.94, 0.94, 0.93])      # o cinza do meio: o zero da escala
    # o polo quente (clay) para valores positivos, o frio (petrol) para negativos
    alvo = np.array([0.659, 0.227, 0.173]) if v >= 0 else np.array([0.122, 0.306, 0.290])
    t = min(abs(v), 1.0)                       # intensidade: |v|, travada em 1
    return tuple(neutro + t * (alvo - neutro)) # interpolação linear entre neutro e o polo

def salvar(fig, nome):
    """Grava a figura em .../fatorial/figuras/ no padrão do projeto (dpi 150)."""
    caminho = CAMINHO_FIG / nome
    # bbox_inches='tight' apara a margem vazia; facecolor mantém o fundo de papel
    fig.savefig(caminho, dpi=150, bbox_inches='tight', facecolor=SURF)
    plt.close(fig)                             # fecha a figura: sem isso a memória cresce
    print(f'figura: {caminho.relative_to(ROOT)}')   # registra o caminho, relativo à raiz

print(f'Raiz do projeto: {ROOT}')              # log: confirma que a raiz foi achada

Raiz do projeto: /Users/pedro/Documents/Iniciacao Cientifica/Projeto_IVS_Censo22


In [2]:
# ── Carga do banco da entrega ────────────────────────────────────────────────
# CD_TIPO entra agora porque o bloco 9 precisa dele (CD_TIPO = 1 marca Favela e
# Comunidade Urbana). Lê-lo aqui evita reabrir o banco lá na frente.
COLUNAS = ['CD_SETOR', 'NM_MUN', 'CD_TIPO', 'urbano', 'Dados_sig',
           'renda_media_sem_extremo'] + IVS7      # identificação + filtros + as 7 do IVS

con = sqlite3.connect(CAMINHO_DB)                 # abre o SQLite (só leitura, na prática)
try:
    # uma consulta só, trazendo apenas as colunas necessárias das 104 da tabela
    bruto = pd.read_sql(f"SELECT {', '.join(COLUNAS)} FROM setores_censitarios", con)
finally:
    con.close()                                   # fecha mesmo se a consulta falhar

# O RECORTE DE ANÁLISE, idêntico ao do Notebook 02: só setor urbano e só setor elegível.
# `urbano` vem como inteiro; astype(str) o normaliza para comparar com '1'.
df = bruto[(pd.to_numeric(bruto['urbano'], errors='coerce') == 1) & (bruto['Dados_sig'] == 'OK')].copy()
df['renda_inv'] = -df['renda_media']          # sentido único: maior = mais vulnerável
# A 2ª rodada da EDA passou a usar a renda sem o setor 310620005650366 de Belo
# Horizonte. A fatorial de agosto é anterior a essa coluna; o bloco 8b mede se a
# troca muda alguma coisa aqui.
df['renda_sem_extremo_inv'] = -df['renda_media_sem_extremo']   # a mesma inversão, na alternativa

# Os dois conjuntos de variáveis que o notebook compara o tempo todo.
IVS7_INV = [c if c != 'renda_media' else 'renda_inv' for c in IVS7]   # 7 componentes
IVS6 = [c for c in IVS7_INV if c != 'pct_lixo_inad']                  # sem o lixo
NOMES7 = [ROTULOS[c] for c in IVS7_INV]       # os rótulos de leitura, na mesma ordem
NOMES6 = [ROTULOS[c] for c in IVS6]           # idem, sem o lixo

# EXCLUSÃO POR LISTA: dropna() derruba o SETOR INTEIRO se faltar qualquer uma das
# variáveis. É a decisão declarada nas limitações, e custa 16.563 setores.
completo7 = df[IVS7_INV].dropna()             # os casos completos nas 7
completo6 = df[IVS6].dropna()                 # e nas 6 — aqui dá o mesmo n, o lixo não falta

print(f'setores no banco            : {len(bruto):,}')          # 109.032, a base inteira
print(f'recorte urbano + Dados_sig OK: {len(df):,}')            # 104.108, o recorte
print(f'completos nas 7 variáveis    : {len(completo7):,}')     # 87.545, o que a fatorial usa
print(f'completos nas 6 (sem lixo)   : {len(completo6):,}')
print(f'perdidos por listwise        : {len(df) - len(completo7):,} '
      f'({100*(len(df)-len(completo7))/len(df):.1f}%)')         # 16.563 = 15,9%

# TRAVA: os dois números que todos os documentos do projeto afirmam. Se não baterem,
# a base mudou e nada abaixo é comparável com o que já foi publicado.
assert len(df) == 104_108, f'recorte deu {len(df):,}, esperado 104.108'
assert len(completo7) == 87_545, f'completos deu {len(completo7):,}, esperado 87.545'
print('\nconferência OK — recorte e casos completos batem com a documentação')

setores no banco            : 109,032
recorte urbano + Dados_sig OK: 104,108
completos nas 7 variáveis    : 87,545
completos nas 6 (sem lixo)   : 87,545
perdidos por listwise        : 16,563 (15.9%)

conferência OK — recorte e casos completos batem com a documentação


## 2. Adequabilidade da base

Etapa 1 do livro (p. 39–46). Reproduz o diagnóstico de 24/08/2026 e acrescenta a **SMC**
por variável, que a p. 42 recomenda como diagnóstico complementar.

**Correlação de Spearman como referência**, com Pearson como sensibilidade. A escolha está
fora do catálogo de correlações do livro (p. 11–15), que cobre combinações de variáveis
categóricas — as sete aqui são contínuas, o que pelo catálogo levaria a Pearson. A
justificativa é a não-normalidade documentada na §9 do relatório da EDA: assimetria de
3,42 na água e 3,74 na renda, curtose de 49,5 na renda. O custo da escolha é grande e está
medido na tabela comparativa abaixo — com Pearson a base **reprovaria** no critério da
maioria dos coeficientes acima de 0,30 (Etapa 1) e ficaria abaixo dos 60% de variância
acumulada (Etapa 2).

**TRAVA:** os números têm de reproduzir `resumo_adequabilidade.csv` exatamente. Se
divergirem, há erro de migração e não se deve seguir.

**Duas leituras a registrar:**

1. O **Bartlett é vazio nesta escala**. A p. 43 adverte que o teste "depende muito do
   tamanho amostral e tende a rejeitar a hipótese nula para amostras grandes". Com 87 mil
   setores, rejeitar H₀ não é evidência de estrutura: é evidência de n. A conclusão de
   adequabilidade se apoia no **KMO e nos MSA individuais**, que não crescem com o n.
2. **Multicolinearidade: o número que circula nos documentos é de outra matriz.** O
   relatório da EDA (§9) e, a partir dele, o guia de leitura e o `GUIA_DO_PROJETO.md`
   citam renda × cor/raça a **−0,811** e o comparam ao limiar de 0,80 da p. 42, acima do
   qual "fica inviável separar o peso delas em cada um dos fatores". Esse −0,811 foi
   calculado **par a par**, cada coeficiente sobre os seus próprios casos completos, nos
   104.108 setores do recorte. A matriz que a análise fatorial decompõe é **listwise**,
   nos 87.545 setores completos nas sete variáveis — e nela o mesmo par dá **0,784**.
   Abaixo do limiar. A diferença é inteiramente o conjunto de setores, e vai na mesma
   direção do viés do sigilo: a base listwise perde os setores de melhor situação, o que
   comprime a associação entre renda e cor/raça. Nenhum par da matriz fatorada chega a
   0,80. Isso não dissolve a objeção — 0,784 continua alto e o bloco socioeconômico
   continua coeso — mas o texto do artigo precisa citar o número da matriz que foi de
   fato fatorada, e dizer qual é qual.

In [3]:
# ── Matrizes de correlação e medidas de adequabilidade ───────────────────────
def adequabilidade(X, nomes, metodo='spearman'):
    """Roda a Etapa 1 do livro sobre um conjunto de variáveis já sem faltantes."""
    R = X.corr(method=metodo).to_numpy()   # a MATRIZ-R: como cada par de variáveis anda junto
    n, p = X.shape                         # n = nº de setores · p = nº de variáveis
    fora = R[~np.eye(p, dtype=bool)]       # só os coeficientes FORA da diagonal (a diagonal é 1)
    kmo_global, msa = kmo(R)               # KMO global e o MSA de cada variável (matriz anti-imagem)
    qui, gl, pval = bartlett(R, n)         # qui-quadrado de Bartlett, graus de liberdade e p
    return {
        'R': pd.DataFrame(R, index=nomes, columns=nomes),   # devolve rotulada, para imprimir
        'n': n, 'p': p, 'razao': n / p,                     # razão casos/variável: o livro pede ≥ 5
        # "a maioria dos coeficientes acima de 0,30" é o critério da p. 44
        'pct_acima_030': float((np.abs(fora) >= 0.30).mean()),
        'kmo': kmo_global, 'msa': pd.Series(msa, index=nomes),
        'smc': pd.Series(smc(R), index=nomes),              # quanto as outras explicam de cada uma
        'qui2': qui, 'gl': gl, 'p_valor': pval,   # 'p' acima é o nº de variáveis
    }

a7s = adequabilidade(completo7, NOMES7, 'spearman')   # o cenário de REFERÊNCIA
a7p = adequabilidade(completo7, NOMES7, 'pearson')    # o mesmo, com Pearson: análise de sensibilidade
a6s = adequabilidade(completo6, NOMES6, 'spearman')   # sem o lixo: a solução recomendada

# TRAVA contra resumo_adequabilidade.csv (gerado em 24/08/2026 a partir do CSV da entrega).
# Se algum destes três falhar, a migração do script para o módulo mudou o resultado —
# e aí NADA abaixo desta célula é comparável com o que já foi publicado.
assert round(a7s['kmo'], 4) == 0.7826, f"KMO deu {a7s['kmo']:.4f}"
assert round(float(a7s['msa'].min()), 4) == 0.6995, f"MSA mín deu {a7s['msa'].min():.4f}"
assert round(a7s['qui2'], 4) == 235084.3838, f"Bartlett deu {a7s['qui2']:.4f}"
print('TRAVA OK — KMO 0,7826 · MSA mín 0,6995 · Bartlett 235.084,3838 reproduzidos\n')

# A tabela comparativa: é ela que mostra o CUSTO de ter escolhido Spearman em vez de Pearson.
comparativo = pd.DataFrame({
    '7 comp. Spearman': [a7s['n'], a7s['razao'], 100*a7s['pct_acima_030'], a7s['kmo'],
                         a7s['msa'].min(), a7s['qui2']],
    '7 comp. Pearson':  [a7p['n'], a7p['razao'], 100*a7p['pct_acima_030'], a7p['kmo'],
                         a7p['msa'].min(), a7p['qui2']],
    '6 comp. Spearman': [a6s['n'], a6s['razao'], 100*a6s['pct_acima_030'], a6s['kmo'],
                         a6s['msa'].min(), a6s['qui2']],
}, index=['n', 'casos/variável', '% |r| >= 0,30', 'KMO', 'MSA mínimo', 'Bartlett qui2'])
print(comparativo.round(3).to_string())   # to_string() imprime a tabela inteira, sem truncar

TRAVA OK — KMO 0,7826 · MSA mín 0,6995 · Bartlett 235.084,3838 reproduzidos

                7 comp. Spearman  7 comp. Pearson  6 comp. Spearman
n                      87545.000        87545.000         87545.000
casos/variável         12506.429        12506.429         14590.833
% |r| >= 0,30             57.143           33.333            80.000
KMO                        0.783            0.732             0.787
MSA mínimo                 0.700            0.542             0.715
Bartlett qui2         235084.384       131592.709        225320.177


In [4]:
# ── SMC, MSA e o cheque de multicolinearidade ────────────────────────────────
# A SMC (p. 42) mede quanto da variabilidade de cada variável as demais explicam. Perto de
# zero indica variável independente das outras — candidata a sair; perto de um indica
# redundância. É a estimativa inicial de comunalidade que o eixo principal usa no bloco 4.
diag7 = pd.DataFrame({'MSA': a7s['msa'], 'SMC': a7s['smc']})   # as duas medidas, por variável
diag6 = pd.DataFrame({'MSA': a6s['msa'], 'SMC': a6s['smc']})   # idem, na solução sem o lixo
print('Sete componentes:');  print(diag7.round(3).to_string())
print('\nSeis componentes (sem lixo):'); print(diag6.round(3).to_string())

# Pares acima do limiar de multicolinearidade da p. 42.
R7 = a7s['R']                                  # a matriz rotulada, para achar o par pelo nome
# varre só o TRIÂNGULO SUPERIOR (j > i): a matriz é simétrica, o resto seria repetição
pares = [(R7.index[i], R7.columns[j], R7.iloc[i, j])
         for i in range(len(R7)) for j in range(i+1, len(R7))
         if abs(R7.iloc[i, j]) >= 0.80]        # |r| ≥ 0,80 é o limiar do livro
print('\nPares com |r| >= 0,80 na matriz FATORADA (listwise, n = 87.545) — limiar da p. 42:')
for a, b, r in pares:
    print(f'  {a} × {b}: {r:.3f}')
if not pares:
    print('  nenhum')                          # e é este o resultado: nenhum par cruza 0,80

# O par que os documentos citam, nas duas matrizes — a divergência é de conjunto de casos,
# não de método, e precisa aparecer com as duas procedências.
par_listwise = R7.loc['Renda (invertida)', 'Cor/raça PPI']     # o valor NA MATRIZ FATORADA
eda = pd.read_csv(ROOT / 'banco_de_dados' / 'eda' / 'correlacao_spearman.csv',
                  sep=';', index_col=0, encoding='utf-8-sig')  # a matriz da EDA, par a par
par_pairwise = abs(eda.loc['renda_media', 'pct_raca_pretpardind'])   # o mesmo par, ali
print(f'\nrenda × cor/raça, em módulo:')
print(f'  matriz fatorada (listwise, n = {len(completo7):,}) : {par_listwise:.3f}')   # 0,784
print(f'  EDA §9 (par a par, recorte de {len(df):,})        : {par_pairwise:.3f}')    # 0,811
print(f'  limiar de multicolinearidade (livro, p. 42)     : 0,800')
print('  -> na matriz que a fatorial decompõe, o par NÃO cruza o limiar')

# Grava o bloco de adequabilidade.
saida_adeq = pd.concat([                       # empilha os dois cenários num CSV só
    diag7.assign(cenario='ivs7_spearman'),     # assign marca de qual cenário é cada linha
    diag6.assign(cenario='ivs6_sem_lixo_spearman'),
]).rename_axis('variavel').reset_index()       # o índice (o nome da variável) vira coluna
# sep=';' e utf-8-sig são o padrão do projeto: assim o Excel em português abre certo
saida_adeq.round(4).to_csv(CAMINHO_FAT / 'nb04_adequabilidade.csv',
                           sep=';', index=False, encoding='utf-8-sig')
a7s['R'].round(3).to_csv(CAMINHO_FAT / 'nb04_correlacao_ivs7_spearman.csv',
                         sep=';', encoding='utf-8-sig')   # a matriz-R, para o deck e o relatório
print('\nnb04_adequabilidade.csv · nb04_correlacao_ivs7_spearman.csv')

Sete componentes:
                      MSA    SMC
Água inadequada     0.753  0.222
Esgoto inadequado   0.834  0.333
Lixo inadequado     0.700  0.106
Razão de moradores  0.857  0.261
Analfabetismo 15+   0.819  0.597
Renda (invertida)   0.724  0.736
Cor/raça PPI        0.782  0.653

Seis componentes (sem lixo):
                      MSA    SMC
Água inadequada     0.745  0.222
Esgoto inadequado   0.850  0.317
Razão de moradores  0.913  0.233
Analfabetismo 15+   0.815  0.597
Renda (invertida)   0.715  0.735
Cor/raça PPI        0.780  0.648

Pares com |r| >= 0,80 na matriz FATORADA (listwise, n = 87.545) — limiar da p. 42:
  nenhum

renda × cor/raça, em módulo:
  matriz fatorada (listwise, n = 87,545) : 0.784
  EDA §9 (par a par, recorte de 104,108)        : 0.811
  limiar de multicolinearidade (livro, p. 42)     : 0,800
  -> na matriz que a fatorial decompõe, o par NÃO cruza o limiar

nb04_adequabilidade.csv · nb04_correlacao_ivs7_spearman.csv


In [5]:
# ── Figura: a matriz-R, as sete variáveis ────────────────────────────────────
# FORMA: correlações são grandezas COM SINAL entre -1 e 1 — paleta DIVERGENTE, dois polos
# opostos com cinza neutro no meio, nunca um arco-íris. O valor vai impresso em cada
# célula: é a codificação secundária que a checagem de cor exige, e é o que permite ler a
# figura impressa em preto e branco. É a Tabela 1 do livro (p. 15) nos dados do projeto.
R7 = a7s['R']
curtos = ['Água', 'Esgoto', 'Lixo', 'Moradores', 'Analfab.', 'Renda (inv.)', 'Cor/raça']  # rótulos curtos

fig, ax = plt.subplots(figsize=(7.6, 6.4), facecolor=SURF)   # um painel só, fundo de papel
M7 = R7.to_numpy()                             # sem os rótulos: só a grade de números
for i in range(len(curtos)):                   # linha
    for j in range(len(curtos)):               # coluna — desenha célula por célula
        v = M7[i, j]
        # cada célula é um retângulo pintado pela rampa; a borda em SURF cria o vão de 2px
        ax.add_patch(plt.Rectangle((j + 0.03, i + 0.03), 0.94, 0.94,
                                   facecolor=rampa_divergente(v), edgecolor=SURF, lw=1.5))
        if i != j:                             # fora da diagonal: imprime o valor
            # − é o sinal de menos tipográfico, e a vírgula é o decimal em português
            ax.text(j + 0.5, i + 0.5, f'{v:.2f}'.replace('-', '−').replace('.', ','),
                    ha='center', va='center', fontsize=9,
                    # texto claro sobre célula escura, escuro sobre clara: garante contraste
                    color=SURF if abs(v) > 0.55 else TINTA)
        else:
            ax.text(j + 0.5, i + 0.5, '1', ha='center', va='center', fontsize=9, color=SURF)
# ylim invertido (7 -> 0) faz a primeira variável ficar no ALTO, como se lê uma tabela
ax.set_xlim(0, len(curtos)); ax.set_ylim(len(curtos), 0)
ax.set_xticks(np.arange(len(curtos)) + 0.5); ax.set_xticklabels(curtos, fontsize=9, color=TINTA)
ax.set_yticks(np.arange(len(curtos)) + 0.5); ax.set_yticklabels(curtos, fontsize=9, color=TINTA)
ax.tick_params(length=0)                       # sem risquinhos de eixo: a grade já orienta
for lado in ('top', 'right', 'left', 'bottom'):
    ax.spines[lado].set_visible(False)         # sem moldura: as células são a própria forma
ax.set_facecolor(SURF)
ax.set_title('Matriz de correlação de Spearman — os sete componentes',
             color=TINTA, fontsize=12, loc='left', pad=14)   # loc='left': título alinhado
# A legenda vai ABAIXO do eixo (y negativo) e diz o que a figura mostra, não como lê-la
fig.text(0.02, -0.035,
         'Tom quente = associação positiva (as duas apontam para mais vulnerabilidade) · '
         'cinza = perto de zero.\nA renda entra invertida. O bloco socioeconômico '
         '(analfabetismo, renda, cor/raça) é o retângulo escuro no canto inferior direito;\n'
         'a linha do lixo é a mais clara da matriz — é a variável que não acompanha nenhuma '
         'outra.',
         color=CINZA, fontsize=8.5, ha='left', linespacing=1.5)
salvar(fig, 'nb04_matriz_correlacao.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_matriz_correlacao.png


## 3. Número de fatores

Três critérios, e o livro (p. 28–32) manda usá-los em conjunto porque costumam discordar.
Aqui eles discordam — e é preciso dizer isso com todas as letras.

**O critério de Kaiser tinha citação a corrigir.** A p. 29 (Stevens) diz que Kaiser é
*mais preciso* quando a amostra passa de 250 casos **e** a comunalidade **média** é igual
ou maior que 0,6 — não a comunalidade mínima de cada variável. Aqui n = 87.545 e a
comunalidade média com 2 fatores é 0,621 (7 variáveis) e 0,700 (6 variáveis): as duas
condições de Stevens são satisfeitas. Tabachnick & Fidell, à parte, recomendam Kaiser
entre 20 e 50 variáveis; aqui são 6 ou 7. É justamente Kaiser quem sustenta o segundo
fator na solução de 7 variáveis, decidindo com folga de 0,09 entre o segundo autovalor
(1,049) e o terceiro (0,958).

**Na solução sem o lixo, Kaiser retém UM fator** — o segundo autovalor é 0,9585, logo
abaixo de 1 — e a análise paralela de Horn concorda. A retenção do segundo fator, o de
saneamento, passa a se apoiar na **razão teórica**: as duas dimensões vêm do IVS-BH 2012.
A p. 32 legitima isso ao dizer que a decisão final pode ser teórica e que a pergunta certa
é "teoricamente faz mais sentido essas variáveis estarem agrupadas em quantos fatores?".
É escolha declarada, não número escondido.

A **análise paralela de Horn** (1965) não está no livro da Enap — está em nota de rodapé em
Figueiredo & Silva (2010). É um critério mais moderno e mais robusto que Kaiser, e o
projeto o usa desde 24/08/2026.

In [6]:
# ── Autovalores, Kaiser, Horn e variância acumulada ──────────────────────────
def espectro(adeq, k=2):
    """Autovalores observados contra os de dados aleatórios de mesmo tamanho (Horn)."""
    R = adeq['R'].to_numpy()               # a matriz de correlação, sem os rótulos
    val, _ = acp(R, k)                     # val = TODOS os autovalores, em ordem decrescente
    # o script original limita a simulação a 20 mil linhas: o custo cai, e o limiar de
    # Horn fica um pouco mais conservador (≈+0,01 no autovalor aleatório médio) — não
    # muda a decisão nos dois cenários. Mantido para reproduzir o que existe.
    hval = horn(min(adeq['n'], 20000), adeq['p'])   # o patamar do acaso, por componente
    return pd.DataFrame({
        'componente': np.arange(1, adeq['p'] + 1),      # 1, 2, 3… um por variável
        'autovalor': val,                               # quanta variância cada componente capta
        # cada autovalor sobre p: com a matriz de correlação, a variância total É p
        'pct_variancia': 100 * val / adeq['p'],
        'pct_acumulado': 100 * np.cumsum(val) / adeq['p'],   # a soma corrida: o critério dos 60%
        'horn_aleatorio': hval,                         # o que dados SEM estrutura produziriam
        'passa_kaiser': val > 1,                        # critério de Kaiser: autovalor acima de 1
        'passa_horn': val > hval,                       # critério de Horn: acima do acaso
    })

esp7, esp6 = espectro(a7s), espectro(a6s)               # os dois cenários, lado a lado
print('Sete componentes (Spearman):'); print(esp7.round(4).to_string(index=False))
print('\nSeis componentes, sem o lixo (Spearman):'); print(esp6.round(4).to_string(index=False))

# O resumo em uma linha por cenário — é aqui que a discordância entre critérios aparece:
# com 7 variáveis os dois retêm 2 fatores; sem o lixo, os dois retêm apenas 1.
for nome, esp in (('7 componentes', esp7), ('6 sem lixo', esp6)):
    print(f"\n{nome}: Kaiser retém {int(esp['passa_kaiser'].sum())} · "
          f"Horn retém {int(esp['passa_horn'].sum())} · "
          f"variância acumulada com 2 fatores = {esp['pct_acumulado'].iloc[1]:.1f}%")
          # .iloc[1] é a SEGUNDA linha: o acumulado até o componente 2

# empilha os dois cenários num CSV só, marcados pela coluna `cenario`
pd.concat([esp7.assign(cenario='ivs7_spearman'), esp6.assign(cenario='ivs6_sem_lixo_spearman')]) \
  .round(4).to_csv(CAMINHO_FAT / 'nb04_autovalores.csv', sep=';', index=False, encoding='utf-8-sig')

Sete componentes (Spearman):
 componente  autovalor  pct_variancia  pct_acumulado  horn_aleatorio  passa_kaiser  passa_horn
          1     3.2997        47.1383        47.1383          1.0259          True        True
          2     1.0486        14.9794        62.1177          1.0155          True        True
          3     0.9575        13.6788        75.7966          1.0075         False       False
          4     0.6198         8.8541        84.6507          1.0005         False       False
          5     0.5544         7.9205        92.5712          0.9927         False       False
          6     0.3478         4.9683        97.5395          0.9848         False       False
          7     0.1722         2.4605       100.0000          0.9731         False       False

Seis componentes, sem o lixo (Spearman):
 componente  autovalor  pct_variancia  pct_acumulado  horn_aleatorio  passa_kaiser  passa_horn
          1     3.2391        53.9858        53.9858          1.0232      

In [7]:
# ── Figura: scree plot com a linha de Horn ───────────────────────────────────
# FORMA: duas séries num eixo só — autovalor observado contra o que dados sem estrutura
# nenhuma produziriam. É *ênfase*, não categórico: o observado é a linha que importa
# (clay, traço cheio, marcador grande), Horn é referência (cinza, tracejado). A linha de
# Kaiser em 1,0 é recessiva, pontilhada. Cada série leva rótulo direto além da legenda —
# quem não distingue as cores lê o rótulo.
fig, eixos = plt.subplots(1, 2, figsize=(11, 4.2), facecolor=SURF)   # dois painéis lado a lado
for ax, esp, titulo in ((eixos[0], esp7, '7 componentes'),
                        (eixos[1], esp6, '6 componentes (sem o lixo)')):
    x = esp['componente']                      # 1, 2, 3… o eixo horizontal
    ax.axhline(1.0, color=CINZA, lw=1, ls=':', zorder=1)   # o corte de Kaiser, recessivo
    ax.plot(x, esp['horn_aleatorio'], color=CINZA, lw=2, ls='--', marker='o',
            ms=5, mfc=SURF, mec=CINZA, zorder=2)           # o acaso: tracejado, marcador vazado
    ax.plot(x, esp['autovalor'], color=CLAY, lw=2, marker='o', ms=8,
            mfc=CLAY, mec=SURF, mew=1.5, zorder=3)         # o observado: cheio, marcador maior
    # zorder crescente empilha na ordem certa: Kaiser atrás, o observado por cima de tudo
    # Os rótulos diretos vão onde as duas linhas estão BEM separadas — no primeiro
    # componente e na cauda. No meio elas se cruzam, e rótulo ali vira colisão.
    ax.annotate('autovalor observado', (x.iloc[0], esp['autovalor'].iloc[0]),
                textcoords='offset points', xytext=(14, -3), color=CLAY, fontsize=9)
    ax.annotate('acaso (Horn)', (x.iloc[4], esp['horn_aleatorio'].iloc[4]),
                textcoords='offset points', xytext=(0, 11), color=CINZA, fontsize=9,
                ha='center')                   # xytext em 'offset points' = deslocamento em pt
    ax.annotate('Kaiser = 1', (x.iloc[-1], 1.0), textcoords='offset points',
                xytext=(-2, -15), color=CINZA, fontsize=8, ha='right')   # na cauda, abaixo
    ax.set_title(titulo, color=TINTA, fontsize=11, loc='left')
    ax.set_xlabel('componente', color=CINZA, fontsize=9)
    ax.set_xticks(x)                           # um risco por componente: são poucos, cabem
    ax.tick_params(colors=CINZA, labelsize=9)
    ax.yaxis.set_major_formatter(VIRGULA)      # 1,0 e não 1.0
    ax.grid(axis='y', color=CINZA, alpha=0.15, lw=0.8)   # grade só na horizontal
    ax.set_axisbelow(True)                     # a grade fica ATRÁS dos dados
    for lado in ('top', 'right'):
        ax.spines[lado].set_visible(False)
    for lado in ('left', 'bottom'):
        ax.spines[lado].set_color(CINZA)
    ax.set_facecolor(SURF)
eixos[0].set_ylabel('autovalor', color=CINZA, fontsize=9)   # rótulo só no painel da esquerda
# Legenda com alças explícitas: passar uma lista de rótulos faria a linha de Kaiser
# entrar como primeira entrada, sem nome, e deslocaria os outros dois.
from matplotlib.lines import Line2D
eixos[0].legend(handles=[                      # Line2D vazio serve só de amostra visual
    Line2D([], [], color=CLAY, lw=2, marker='o', ms=7, mfc=CLAY, mec=SURF, mew=1.2,
           label='autovalor observado'),
    Line2D([], [], color=CINZA, lw=2, ls='--', marker='o', ms=5, mfc=SURF, mec=CINZA,
           label='acaso (Horn)'),
], frameon=False, fontsize=8.5, labelcolor=TINTA,   # frameon=False: sem caixa em volta
    # longe do rótulo direto do primeiro componente, que fica no alto à esquerda
    loc='center right', bbox_to_anchor=(1.0, 0.63))
fig.suptitle('Quantos fatores reter — observado contra o acaso',
             color=TINTA, fontsize=12, x=0.007, ha='left', y=1.02)   # título da figura inteira
salvar(fig, 'nb04_scree_horn.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_scree_horn.png


## 4. Extração comparada — ACP e eixo principal

A p. 27 traz duas regras sobre quando ACP e análise fatorial dão a mesma coisa: Hair
(acima de 30 variáveis, **ou** comunalidades acima de 0,60 na maioria) e Stevens (com 30 ou
mais variáveis e comunalidades acima de 0,7 em todas, as soluções ficam muito próximas;
**abaixo de 20 variáveis e com comunalidades baixas, abaixo de 0,4, podem divergir**).

O projeto tem 6 variáveis, abaixo do patamar de Hair e de Stevens — mas 5 das 6
comunalidades passam de 0,60 (0,822; 0,615; 0,770; 0,856; 0,754), o que satisfaz a
condição de maioria de Hair. Pela regra de Hair esperava-se convergência; o teste abaixo
mostra que, empiricamente, ela falha na água — "ACP e AF dariam o mesmo" não é
pressuposto: é **hipótese testada**, e caiu.

A diferença entre as duas está inteiramente na diagonal da matriz decomposta: a ACP põe 1,
usando toda a variância de cada variável; a análise fatorial põe a comunalidade, usando só
a variância compartilhada. Como a comunalidade só se conhece depois de extrair, o eixo
principal itera a partir da SMC.

In [8]:
# ── ACP e fatoração do eixo principal, lado a lado ───────────────────────────
def comparar_extracao(adeq, nomes, k=2):
    R = adeq['R'].to_numpy()
    _, c_acp = acp(R, k)                          # cargas pela ACP: diagonal 1, toda a variância
    c_paf, h_paf, info = fatoracao_eixo_principal(R, k)   # pelo eixo principal: só a compartilhada
    # o sinal do autovetor é arbitrário: alinha o PAF ao ACP coluna a coluna antes de
    # comparar, senão a diferença mediria troca de sinal, não divergência de método.
    for j in range(k):
        if c_paf[:, j] @ c_acp[:, j] < 0:         # produto interno negativo = sentidos opostos
            c_paf[:, j] = -c_paf[:, j]            # vira a coluna inteira
    tab = pd.DataFrame({
        # ** desempacota o dicionário: gera ACP_1, ACP_2, … sem escrever um por um
        **{f'ACP_{j+1}': c_acp[:, j] for j in range(k)},
        **{f'PAF_{j+1}': c_paf[:, j] for j in range(k)},
        **{f'dif_{j+1}': np.abs(c_acp[:, j] - c_paf[:, j]) for j in range(k)},   # a divergência
        'comun_ACP': (c_acp ** 2).sum(axis=1),    # comunalidade = soma dos quadrados das cargas
        'comun_PAF': h_paf,                       # a do eixo principal vem da iteração
        'SMC': adeq['smc'].to_numpy(),            # e a SMC, para ler a divergência contra ela
    }, index=nomes)
    return tab, info                              # info traz iterações, delta e caso de Heywood

ext6, info6 = comparar_extracao(a6s, NOMES6)      # solução recomendada, sem o lixo
ext7, info7 = comparar_extracao(a7s, NOMES7)      # e a de sete, onde o lixo ainda está

for nome, tab, info in (('6 componentes (sem lixo)', ext6, info6),
                        ('7 componentes', ext7, info7)):
    difs = tab[[c for c in tab.columns if c.startswith('dif_')]]   # só as colunas de diferença
    pior = difs.stack().idxmax()                  # stack() empilha; idxmax devolve (linha, coluna)
    print(f'\n── {nome} ──')
    print(tab.round(3).to_string())
    print(f"convergiu em {info['iteracoes']} iterações (delta {info['delta']:.2e}) · "
          f"caso de Heywood: {info['heywood']}")  # Heywood = comunalidade acima de 1, inadmissível
    print(f'maior diferença absoluta: {difs.to_numpy().max():.3f} '
          f'em {pior[0]} / {pior[1]}')            # onde as duas técnicas mais discordam

# LEITURA: a maior divergência é a da água, que tem a SMC mais baixa; a menor é a da
# renda, que tem a mais alta. O resto não segue a SMC nessa ordem (razão de moradores,
# SMC 0,233, tem a 2ª menor divergência; cor/raça, SMC 0,648, diverge mais que o esgoto).
pd.concat([ext6.assign(cenario='ivs6_sem_lixo_spearman'),
           ext7.assign(cenario='ivs7_spearman')]).rename_axis('variavel') \
  .round(4).to_csv(CAMINHO_FAT / 'nb04_extracao_comparada.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_extracao_comparada.csv')


── 6 componentes (sem lixo) ──
                    ACP_1  ACP_2  PAF_1  PAF_2  dif_1  dif_2  comun_ACP  comun_PAF    SMC
Água inadequada    -0.503  0.754 -0.432  0.513  0.071  0.241      0.822      0.450  0.222
Esgoto inadequado  -0.666  0.413 -0.576  0.315  0.090  0.098      0.615      0.431  0.317
Razão de moradores -0.616  0.023 -0.503  0.086  0.113  0.062      0.380      0.260  0.233
Analfabetismo 15+  -0.825 -0.299 -0.783 -0.159  0.042  0.140      0.770      0.638  0.597
Renda (invertida)  -0.871 -0.312 -0.912 -0.293  0.041  0.019      0.856      0.918  0.735
Cor/raça PPI       -0.850 -0.178 -0.814 -0.067  0.036  0.111      0.754      0.666  0.648
convergiu em 192 iterações (delta 9.93e-08) · caso de Heywood: False
maior diferença absoluta: 0.241 em Água inadequada / dif_2

── 7 componentes ──
                    ACP_1  ACP_2  PAF_1  PAF_2  dif_1  dif_2  comun_ACP  comun_PAF    SMC
Água inadequada    -0.501  0.049 -0.421  0.445  0.080  0.396      0.253      0.376  0.222
Esgoto in

## 5. Rotação comparada — Varimax e promax

**A passagem mais consequente do livro para este projeto está na p. 38:** "usar rotação
ortogonal com dados de Ciências Humanas e Sociais não parece ter nenhum sentido. Nessas
áreas, as variáveis quase sempre são correlacionadas. [...] para usar rotação ortogonal, o
pesquisador precisaria ter evidências teóricas ou empíricas muito fortes de que os fatores
não são correlacionados."

A Varimax usada até aqui foi herdada de Figueiredo & Silva (2010), que a adota "por ser a
mais comum". O livro inverte o ônus da prova: ortogonal é o caminho que exige
justificativa, e a justificativa pedida é de que os fatores **não** se correlacionam. No
IVS isso é implausível à partida — territórios pobres têm pior saneamento, e as próprias
correlações da matriz fatorada mostram: esgoto com analfabetismo a 0,429, esgoto com
renda (invertida) a 0,436 — os dois pela matriz que a análise decompõe, não pela EDA par
a par.

A rotação oblíqua produz três coisas que a ortogonal não pode produzir:

1. **Φ, a matriz de correlação entre os fatores** — evidência de validação que o projeto
   hoje não tem. Se os dois fatores se correlacionarem positivamente e com magnitude
   moderada, é o que a teoria da vulnerabilidade prevê.
2. **Duas matrizes de cargas** (p. 21–22): a **padrão**, de coeficientes de regressão, e a
   **estrutura**, de correlações. O livro registra que a maioria interpreta a padrão, e é
   dela que saem os pesos. Na solução ortogonal as duas coincidem.
3. **A repartição de pesos não é mais uma conta só.** O 65/35 ortogonal foi calculado por
   soma dos quadrados de cargas. Com fatores correlacionados, essa soma deixa de ser uma
   partição aditiva: pela matriz padrão dá um número, pela matriz estrutura dá outro, e
   nenhum dos dois é "o" correto — os dois entram na tabela abaixo lado a lado.

**Cheque obrigatório (p. 22):** numa solução oblíqua uma carga pode passar de 1 sem ser
erro, por ser coeficiente de regressão. Se isso ocorrer, testa-se a variância residual da
variável: negativa, a solução é inadmissível e sugere fatores demais extraídos.

In [9]:
# ── Varimax × promax ─────────────────────────────────────────────────────────
def comparar_rotacao(adeq, nomes, k=2):
    R = adeq['R'].to_numpy()
    _, cargas = acp(R, k)                    # o ponto de partida: cargas SEM rotação
    vmax = varimax(cargas)                   # rotação ORTOGONAL: supõe fatores independentes
    padrao, estrutura, phi = rotacao_promax(cargas)   # OBLÍQUA: deixa os fatores se correlacionarem
    # padrão = coeficientes de regressão (de onde saem os pesos)
    # estrutura = correlações variável × fator · phi = correlação ENTRE os fatores

    # Convenção de leitura: o sinal do autovetor é arbitrário. Vira-se cada fator para que
    # a soma das cargas fique positiva — assim "carga positiva = mais vulnerável", que é o
    # sentido em que as variáveis já foram postas (renda invertida).
    for j in range(k):
        if vmax[:, j].sum() < 0:             # se o fator saiu com o sentido invertido…
            vmax[:, j] *= -1                 # …vira a coluna inteira
    sinais = np.where(padrao.sum(axis=0) < 0, -1.0, 1.0)   # +1 ou -1 para cada fator
    padrao = padrao * sinais                 # aplica o sinal às cargas padrão
    estrutura = estrutura * sinais           # e às de estrutura, para não descasar
    phi = phi * np.outer(sinais, sinais)     # Φ precisa do sinal nos DOIS lados: sᵢ × sⱼ

    comun_ort = (vmax ** 2).sum(axis=1)      # ortogonal: comunalidade é a soma dos quadrados
    # OBLÍQUA é diferente: parte da variância é compartilhada entre os fatores
    # correlacionados e seria contada duas vezes — a conta certa é diag(padrão·Φ·padrãoᵀ).
    comun_obl = comunalidades_obliquas(padrao, phi)
    tab = pd.DataFrame({
        **{f'Varimax{j+1}': vmax[:, j] for j in range(k)},
        **{f'Padrao{j+1}': padrao[:, j] for j in range(k)},
        **{f'Estrutura{j+1}': estrutura[:, j] for j in range(k)},
        'comun_ortogonal': comun_ort,
        'comun_obliqua': comun_obl,
        'var_residual': 1 - comun_obl,       # o que os fatores NÃO explicam — o teste da p. 22
    }, index=nomes)

    def repartir(M):
        """Quanto cada fator pesa: soma dos quadrados das cargas, normalizada."""
        ss = (M ** 2).sum(axis=0)            # soma por COLUNA = por fator
        return ss / ss.sum()                 # em proporção do total

    return tab, phi, repartir(vmax), repartir(padrao), repartir(estrutura)

rot6, phi6, rep6_ort, rep6_obl, rep6_est = comparar_rotacao(a6s, NOMES6)   # a solução recomendada
rot7, phi7, rep7_ort, rep7_obl, rep7_est = comparar_rotacao(a7s, NOMES7)   # e a de sete variáveis

print('── 6 componentes, sem o lixo ──')
print(rot6.round(3).to_string())
# Φ é o objeto que a rotação ortogonal NÃO pode produzir. Fora da diagonal, dá 0,522:
# os dois fatores andam juntos, que é o que a teoria da vulnerabilidade prevê.
print(f'\nΦ (correlação entre os fatores):\n{pd.DataFrame(phi6).round(3).to_string()}')
print(f'\nrepartição do peso   ortogonal: {100*rep6_ort[0]:.1f} / {100*rep6_ort[1]:.1f}')
print(f'repartição oblíqua (padrão)   : {100*rep6_obl[0]:.1f} / {100*rep6_obl[1]:.1f}')
print(f'repartição oblíqua (estrutura): {100*rep6_est[0]:.1f} / {100*rep6_est[1]:.1f}')
print(f'referência IVS-BH 2012        : 60,0 / 40,0')   # a literatura, para comparar
# nenhuma das duas repartições oblíquas é partição aditiva com fatores correlacionados;
# nenhuma delas "afasta" ou "aproxima" oficialmente dos 60/40 — ficam lado a lado (NB4-03)

# Cheque da p. 22: carga acima de 1 exige variância residual positiva.
# Numa solução oblíqua a carga é COEFICIENTE DE REGRESSÃO, e pode passar de 1 sem ser erro.
altas = rot6[[c for c in rot6.columns if c.startswith('Padrao')]].abs().to_numpy().max()
print(f'\nmaior carga padrão em módulo: {altas:.3f}')
if altas > 1:
    print('  carga acima de 1 — conferindo a variância residual (livro p. 22):')
    print(rot6[['var_residual']].round(4).to_string())
    # residual NEGATIVA significa solução inadmissível, e sugere fatores demais extraídos
    assert (rot6['var_residual'] > 0).all(), 'variância residual negativa: solução inadmissível'
    print('  todas positivas — solução admissível')
else:
    print('  nenhuma carga acima de 1; o cheque da p. 22 não se aplica')
# a trava vale para os DOIS cenários, mesmo quando nenhuma carga passou de 1
assert (rot6['var_residual'] > 0).all() and (rot7['var_residual'] > 0).all()

print('\n── 7 componentes ──')
print(rot7.round(3).to_string())
print(f'\nΦ:\n{pd.DataFrame(phi7).round(3).to_string()}')   # aqui Φ é bem menor: 0,151
print(f'repartição ortogonal: {100*rep7_ort[0]:.1f} / {100*rep7_ort[1]:.1f} · '
      f'oblíqua padrão: {100*rep7_obl[0]:.1f} / {100*rep7_obl[1]:.1f} · '
      f'oblíqua estrutura: {100*rep7_est[0]:.1f} / {100*rep7_est[1]:.1f}')

rot6.rename_axis('variavel').round(4).to_csv(     # as cargas da solução recomendada
    CAMINHO_FAT / 'nb04_cargas_ivs6_sem_lixo.csv', sep=';', encoding='utf-8-sig')
rot7.rename_axis('variavel').round(4).to_csv(     # e as de sete, para comparação
    CAMINHO_FAT / 'nb04_cargas_ivs7.csv', sep=';', encoding='utf-8-sig')
pd.DataFrame(phi6, index=['Fator 1', 'Fator 2'], columns=['Fator 1', 'Fator 2']) \
  .round(4).to_csv(CAMINHO_FAT / 'nb04_phi_ivs6_sem_lixo.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_cargas_ivs6_sem_lixo.csv · nb04_cargas_ivs7.csv · nb04_phi_ivs6_sem_lixo.csv')

── 6 componentes, sem o lixo ──
                    Varimax1  Varimax2  Padrao1  Padrao2  Estrutura1  Estrutura2  comun_ortogonal  comun_obliqua  var_residual
Água inadequada        0.086     0.903   -0.211    0.999       0.310       0.889            0.822          0.822         0.178
Esgoto inadequado      0.392     0.679    0.207    0.656       0.549       0.764            0.615          0.615         0.385
Razão de moradores     0.532     0.312    0.490    0.198       0.593       0.454            0.380          0.380         0.620
Analfabetismo 15+      0.868     0.127    0.930   -0.111       0.872       0.374            0.770          0.770         0.230
Renda (invertida)      0.915     0.137    0.979   -0.113       0.920       0.398            0.856          0.856         0.144
Cor/raça PPI           0.833     0.245    0.850    0.034       0.868       0.477            0.754          0.754         0.246

Φ (correlação entre os fatores):
       0      1
0  1.000  0.522
1  0.522  1.0

In [10]:
# ── Figura: mapa de cargas ───────────────────────────────────────────────────
# FORMA: as cargas são grandezas COM SINAL, entre -1 e 1 — logo, paleta DIVERGENTE:
# dois polos opostos (clay quente = positivo, petrol frio = negativo) com cinza neutro no
# meio. Nunca um arco-íris, nunca uma cor no ponto médio. E cada célula leva o valor
# impresso: é a codificação secundária que a validação de cor exige, e quem lê em preto e
# branco continua lendo a tabela.
fig, eixos = plt.subplots(1, 2, figsize=(9.0, 4.2), facecolor=SURF,
                          gridspec_kw={'width_ratios': [1, 1]})   # os dois painéis iguais
for ax, M, titulo in (
        (eixos[0], rot6[['Varimax1', 'Varimax2']], 'Varimax (ortogonal)'),
        (eixos[1], rot6[['Padrao1', 'Padrao2']], 'promax — matriz padrão (oblíqua)')):
    dados = M.to_numpy()                       # 6 variáveis × 2 fatores
    for i in range(dados.shape[0]):            # linha = variável
        for j in range(dados.shape[1]):        # coluna = fator
            v = dados[i, j]
            ax.add_patch(plt.Rectangle((j + 0.02, i + 0.02), 0.96, 0.96,
                                       facecolor=rampa_divergente(v), edgecolor=SURF, lw=2))
            ax.text(j + 0.5, i + 0.5, f'{v:.2f}'.replace('-', '−').replace('.', ','),
                    ha='center', va='center', fontsize=9.5,
                    color=SURF if abs(v) > 0.55 else TINTA)   # contraste garantido
    ax.set_xlim(0, dados.shape[1]); ax.set_ylim(dados.shape[0], 0)   # invertido: 1ª linha no alto
    ax.set_xticks(np.arange(dados.shape[1]) + 0.5)      # +0,5 centraliza o rótulo na célula
    ax.set_xticklabels(['Fator 1', 'Fator 2'], fontsize=9, color=TINTA)
    ax.set_yticks(np.arange(dados.shape[0]) + 0.5)
    # os nomes das variáveis só no painel da esquerda: à direita seriam repetição
    ax.set_yticklabels(M.index if ax is eixos[0] else [], fontsize=9, color=TINTA)
    ax.tick_params(length=0)
    for lado in ('top', 'right', 'left', 'bottom'):
        ax.spines[lado].set_visible(False)
    ax.set_title(titulo, color=TINTA, fontsize=11, loc='left')
    ax.set_facecolor(SURF)
fig.suptitle('Cargas fatoriais — seis componentes, sem o indicador de lixo',
             color=TINTA, fontsize=12, x=0.007, ha='left', y=1.03)
phi_txt = f'{phi6[0, 1]:.3f}'.replace('.', ',')   # Φ com vírgula decimal, para a legenda
fig.text(0.007, -0.05,
         'Tom quente = carga positiva · tom frio = carga negativa · cinza = perto de zero.\n'
         'O valor vai impresso em cada célula: a leitura não depende de distinguir as cores.  '
         f'Correlação entre os fatores (Φ) = {phi_txt}',
         color=CINZA, fontsize=8.5, ha='left', linespacing=1.6)
salvar(fig, 'nb04_mapa_cargas.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_mapa_cargas.png


### O plano dos fatores

A Figura 2 do livro (p. 17) desenha os fatores como **eixos de um sistema de coordenadas**,
com cada variável plotada nas suas duas cargas. Os eixos vão de −1 a 1, que são os limites
do coeficiente de correlação, e a proximidade entre pontos indica variáveis que caminham
juntas.

O exemplo do livro tem um item — o item 7 — que "apresenta correlações baixas com todas as
outras" e que, no gráfico, "se encontra espacialmente distante dos dois grupos". O projeto
tem exatamente essa figura, e ela tem nome: o indicador de **lixo**. Vale ver os dois
painéis lado a lado, porque o segundo mostra o que acontece com a estrutura quando ele sai.

In [11]:
# ── Figura: representação gráfica dos fatores (livro, Figura 2, p. 17) ───────
# FORMA: dispersão em que a POSIÇÃO é a informação — as duas cargas de cada variável. Cor
# não carrega identidade aqui (o rótulo direto carrega); ela marca uma coisa só, a
# variável que não pertence ao construto. É a forma de ÊNFASE: um ponto no acento, o resto
# recessivo. Eixos de -1 a 1, como o livro fixa, e não ajustados aos dados: encolher os
# eixos faria cargas médias parecerem altas.
def posicionar_rotulos(pontos, textos, lim=1.0):
    '''Escolhe, para cada rótulo, a direção que menos colide.

    As seis variáveis socioeconômicas caem quase no mesmo ponto do plano — é o próprio
    achado — e rótulo fixo em cima delas fica ilegível. Para cada ponto, testa oito
    direções em dois raios e fica com a que menos sobrepõe rótulo já posto, ponto do
    gráfico, ou borda. É guloso e basta para sete itens.
    '''
    alt, larg_car = 0.085, 0.036          # tamanho aproximado do texto em unidades do eixo
    direcoes = [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]  # 8 direções
    postos_ = []                          # os retângulos dos rótulos já colocados
    for (px, py), txt in zip(pontos, textos):
        larg = larg_car * len(txt)        # largura estimada pelo nº de caracteres
        melhor, melhor_custo = None, np.inf
        for raio in (0.11, 0.20):         # tenta perto; se não couber, mais longe
            for dx, dy in direcoes:
                norma = np.hypot(dx, dy) or 1     # normaliza a diagonal (√2), não a reta
                cx = px + raio * dx / norma + (larg / 2) * np.sign(dx)   # centro do rótulo
                cy = py + raio * dy / norma + (alt / 2) * (dy if dy else 0.6)
                custo = 0.0               # quanto MENOR o custo, melhor a posição
                # sair do quadro é o pior defeito: o rótulo some
                custo += 300 * max(0, (cx + larg / 2) - lim) + 300 * max(0, -lim - (cx - larg / 2))
                custo += 300 * max(0, (cy + alt / 2) - lim) + 300 * max(0, -lim - (cy - alt / 2))
                for (qx, qy, ql, qa) in postos_:      # sobreposição com rótulo já posto
                    # a área de interseção dos dois retângulos, em cada eixo
                    ox = max(0, min(cx + larg/2, qx + ql/2) - max(cx - larg/2, qx - ql/2))
                    oy = max(0, min(cy + alt/2, qy + qa/2) - max(cy - alt/2, qy - qa/2))
                    custo += 40 * ox * oy             # penaliza pela ÁREA sobreposta
                for (ox_, oy_) in pontos:             # não cobrir marca de dado
                    if abs(ox_ - cx) < larg/2 and abs(oy_ - cy) < alt/2:
                        custo += 3
                if abs(cy) < alt / 2 or abs(cx) < 0.02:   # nem cair em cima dos eixos zero
                    custo += 2
                custo += 0.5 * raio                   # perto é melhor, se empatar
                if custo < melhor_custo:              # guarda a melhor posição até agora
                    melhor, melhor_custo = (cx, cy, larg, alt, dx, dy), custo
        postos_.append(melhor[:4])        # o rótulo escolhido entra na lista de obstáculos
        yield melhor                      # gerador: devolve um por vez, na ordem dos pontos


def plano_fatorial(ax, cargas, nomes, destaque=None, titulo='', rotulo_y2='Fator 2 — saneamento'):
    ax.axhline(0, color=CINZA, lw=0.9, zorder=1)   # os dois eixos do zero, que dividem
    ax.axvline(0, color=CINZA, lw=0.9, zorder=1)   # o plano em quadrantes
    pontos = [(cargas[i, 0], cargas[i, 1]) for i in range(len(nomes))]   # (carga F1, carga F2)
    colocados = list(posicionar_rotulos(pontos, nomes))   # resolve as colisões de uma vez
    for i, nome in enumerate(nomes):
        x, y = pontos[i]
        fora = (nome == destaque)         # a variável em destaque (o lixo) recebe o acento
        cor = CLAY if fora else PETROL
        cx, cy, larg, alt, dx, dy = colocados[i]
        # fio fino ligando a marca ao rótulo quando ele precisou se afastar
        if np.hypot(cx - x, cy - y) > 0.16:
            ax.plot([x, cx - np.sign(dx) * larg / 2 if dx else cx], [y, cy],
                    color=cor, lw=0.7, alpha=0.5, zorder=2)
        ax.scatter([x], [y], s=95 if fora else 70, color=cor, zorder=4,
                   edgecolor=SURF, linewidth=1.4)   # a borda clara separa pontos vizinhos
        ax.text(cx, cy, nome, ha='center', va='center', fontsize=9.5, color=cor,
                fontweight='bold' if fora else 'normal', zorder=5)   # negrito só no destaque
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)   # FIXOS: são os limites do coeficiente de correlação
    ax.set_xticks([-1, -0.5, 0, 0.5, 1]); ax.set_yticks([-1, -0.5, 0, 0.5, 1])
    ax.xaxis.set_major_formatter(VIRGULA); ax.yaxis.set_major_formatter(VIRGULA)
    ax.tick_params(colors=CINZA, labelsize=8.5)
    ax.set_xlabel('Fator 1 — socioeconômico', color=CINZA, fontsize=9.5)
    ax.set_ylabel(rotulo_y2, color=CINZA, fontsize=9.5)
    ax.set_title(titulo, color=TINTA, fontsize=11, loc='left')
    ax.set_aspect('equal')                # 1:1 — sem isso a distância entre pontos engana
    ax.grid(color=CINZA, alpha=0.12, lw=0.7); ax.set_axisbelow(True)
    for lado in ('top', 'right'):
        ax.spines[lado].set_visible(False)
    for lado in ('left', 'bottom'):
        ax.spines[lado].set_color(CINZA)
    ax.set_facecolor(SURF)

fig, eixos = plt.subplots(1, 2, figsize=(12.2, 6.0), facecolor=SURF)
plano_fatorial(eixos[0], rot7[['Varimax1', 'Varimax2']].to_numpy(), NOMES7,
               destaque='Lixo inadequado', titulo='Sete componentes',
               rotulo_y2='Fator 2 — lixo')   # aqui o lixo DEFINE o fator sozinho
plano_fatorial(eixos[1], rot6[['Varimax1', 'Varimax2']].to_numpy(), NOMES6,
               titulo='Seis componentes, sem o lixo')                   # e sem ele
fig.suptitle('O plano dos fatores — cada variável nas suas duas cargas',
             color=TINTA, fontsize=12.5, x=0.007, ha='left', y=1.0)
fig.text(0.007, -0.02,
         'Eixos de −1 a 1, os limites do coeficiente de correlação, como na Figura 2 do '
         'livro (p. 17). Pontos próximos são variáveis que caminham juntas.\n'
         'À esquerda, o lixo ocupa sozinho o alto do eixo vertical: ao contrário do item 7 '
         'do livro, que "não pertence a nenhum fator", aqui o lixo DEFINE um fator '
         'próprio — o fator 2 desta solução.',
         color=CINZA, fontsize=8.5, ha='left', linespacing=1.5)
salvar(fig, 'nb04_plano_fatorial.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_plano_fatorial.png


## 6. Estabilidade das cargas — bootstrap

> **Nota (set/2026):** a incerteza amostral deste projeto passou a ser reportada também
> por sensibilidade a município (deixar um de fora por vez), no notebook 04b, a pedido
> da orientadora. Este bloco de bootstrap por setor continua de pé, sem alterações; ele
> reamostra setores como se fossem independentes, o que a limitação 5 abaixo já registra
> que não são.

A p. 23 acusa os métodos não refinados de serem "muito instável[eis] por depender
fortemente da amostra em particular que está sendo analisada". É uma crítica séria ao
plano do projeto, que prevê o IVS como média ponderada. A resposta honesta não é discordar:
é medir.

Mil reamostragens com reposição, refazendo a matriz de correlação, a extração e a rotação
a cada uma. Cada solução é alinhada à da amostra completa antes de entrar na conta — sem
isso, a troca de sinal ou de ordem dos fatores que o LAPACK faz a cada rodada produziria
cancelamento, e o resultado pareceria instabilidade sendo artefato.

Nem a Enap nem Figueiredo trazem bootstrap. É extensão do projeto.

In [12]:
# ── Bootstrap das cargas e da repartição ─────────────────────────────────────
X6 = completo6.to_numpy()          # só os números: a reamostragem não precisa dos rótulos
# n_rep=1000 reamostragens COM REPOSIÇÃO. A cada uma: sorteia 87.545 setores, refaz a
# matriz de Spearman, a extração e a rotação. seed=42 fixa o sorteio — sem isso o
# resultado mudaria a cada execução e o CSV não seria reprodutível. Leva ~1,3 min.
bs = bootstrap_cargas(X6, k=2, n_rep=1000, seed=42, metodo='spearman', rotacao='varimax')

lo_c, hi_c = bs['cargas_ic']       # os percentis 2,5 e 97,5 de cada carga: o IC de 95%
ic_cargas = pd.DataFrame({
    'carga_F1': bs['cargas'][:, 0], 'IC95_F1_inf': lo_c[:, 0], 'IC95_F1_sup': hi_c[:, 0],
    'largura_F1': hi_c[:, 0] - lo_c[:, 0],      # a largura é a leitura direta: quanto oscila
    'carga_F2': bs['cargas'][:, 1], 'IC95_F2_inf': lo_c[:, 1], 'IC95_F2_sup': hi_c[:, 1],
    'largura_F2': hi_c[:, 1] - lo_c[:, 1],
}, index=NOMES6)
print(ic_cargas.round(4).to_string())

lo_p, hi_p = bs['repartição_ic']   # o mesmo IC, agora para a repartição ENTRE as dimensões
print(f"\nrepartição do peso entre as dimensões (1.000 reamostragens, semente 42):")
print(f"  Fator 1: {100*bs['repartição'][0]:.1f}%  IC95 [{100*lo_p[0]:.1f}; {100*hi_p[0]:.1f}]")
print(f"  Fator 2: {100*bs['repartição'][1]:.1f}%  IC95 [{100*lo_p[1]:.1f}; {100*hi_p[1]:.1f}]")
# 0,59 ponto percentual: mede a oscilação por reamostragem de SETORES (iid). Não cobre
# a dependência espacial entre municípios — ver a sensibilidade por município no 04b.
print(f"  amplitude do IC: {100*(hi_p[0]-lo_p[0]):.2f} pontos percentuais")

ic_cargas.rename_axis('variavel').round(4).to_csv(
    CAMINHO_FAT / 'nb04_bootstrap_cargas.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_bootstrap_cargas.csv')

                    carga_F1  IC95_F1_inf  IC95_F1_sup  largura_F1  carga_F2  IC95_F2_inf  IC95_F2_sup  largura_F2
Água inadequada      -0.0864      -0.0904      -0.0823      0.0081    0.9026       0.8991       0.9055      0.0064
Esgoto inadequado    -0.3917      -0.3998      -0.3837      0.0161    0.6794       0.6705       0.6879      0.0174
Razão de moradores   -0.5320      -0.5446      -0.5185      0.0261    0.3121       0.2935       0.3340      0.0405
Analfabetismo 15+    -0.8683      -0.8705      -0.8660      0.0045    0.1271       0.1215       0.1335      0.0119
Renda (invertida)    -0.9151      -0.9170      -0.9132      0.0038    0.1374       0.1327       0.1421      0.0093
Cor/raça PPI         -0.8328      -0.8353      -0.8302      0.0051    0.2451       0.2394       0.2514      0.0119

repartição do peso entre as dimensões (1.000 reamostragens, semente 42):
  Fator 1: 65.0%  IC95 [64.7; 65.3]
  Fator 2: 35.0%  IC95 [34.7; 35.3]
  amplitude do IC: 0.57 pontos percentuais

nb04_

## 7. Pesos e escores

Dois caminhos para transformar cargas em índice, e o livro é explícito sobre a diferença
(p. 22–26):

- **Índice 0–1 por média ponderada** das variáveis padronizadas. É o plano do projeto e é
  o que o IVS-BH 2012 faz. Na taxonomia da p. 23 é um método **não refinado**, com três
  defeitos declarados: depende da escala das variáveis, depende da rotação, e é instável
  entre amostras.
- **Escore pelo método da regressão**, B = R⁻¹A (p. 25). Refinado, estável, mas sai numa
  escala padronizada abstrata, perde a comparabilidade com o IVS-BH e exige a matriz R⁻¹
  para ser reproduzido por terceiros.

A saída não é escolher, é **calcular os dois e usar a concordância como validação**. Se os
dois ordenarem os setores do mesmo jeito, a instabilidade que o livro teme não se
materializou nesta amostra, e o índice 0–1 fica reportado com a evidência no apêndice.

**Composição dos pesos.** Cada variável é atribuída à dimensão em que carrega mais alto; o
peso da dimensão é a soma dos quadrados das cargas daquele fator sobre o total; dentro da
dimensão, o peso de cada variável é proporcional ao quadrado da sua carga. É a formulação
que a p. 71 defende: "os itens contribuem de maneira desigual para o fator: quanto maior a
carga fatorial, maior a contribuição do item", ao contrário das técnicas mais simples que
pressupõem contribuição igual.

**A padronização aqui é min-max global, e é provisória.** A normalização definitiva é
**por município** e pertence ao Notebook 03. Está medido que normalizar antes de fatorar
derruba o KMO de 0,783 para 0,720 e muda a repartição de 65/35 para 56/44 — por isso a
fatorial roda sobre os brutos. O índice calculado aqui serve para comparar cenários, não
é o IVS final.

In [13]:
# ── Dos carregamentos aos pesos ──────────────────────────────────────────────
def montar_pesos(cargas_df, colunas, nomes, rep):
    """Atribui cada variável à dimensão de maior carga e reparte o peso dentro dela."""
    M = cargas_df.to_numpy()
    dimensao = np.abs(M).argmax(axis=1)                 # a qual fator cada variável pertence
    peso = np.zeros(len(nomes))                         # começa tudo em zero
    for j in range(M.shape[1]):                         # para cada fator…
        membros = dimensao == j                         # …quais variáveis pertencem a ele
        if not membros.any():                           # fator sem nenhuma variável: pula
            continue
        dentro = M[membros, j] ** 2                     # o quadrado da carga = a contribuição
        peso[membros] = rep[j] * dentro / dentro.sum()   # peso da dimensão × parte na dimensão
    return pd.DataFrame({'coluna': colunas, 'dimensao': dimensao + 1,   # +1: fator 1 e 2, não 0 e 1
                         'carga': M[np.arange(len(nomes)), dimensao],   # a carga na dimensão dela
                         'peso': peso}, index=nomes)

def vetor(pesos_df, colunas):
    """Os pesos na ordem das colunas, casados por NOME.

    O índice é uma soma ponderada: se um peso se juntar à variável errada, nada falha e o
    número sai errado — o pior defeito possível nesta etapa. Casar por nome custa uma
    linha e fecha essa porta.
    """
    return pesos_df.set_index('coluna').loc[list(colunas), 'peso'].to_numpy()


# A SOLUÇÃO OFICIAL: Varimax, 6 componentes, repartição empírica entre as dimensões.
pesos = montar_pesos(rot6[['Varimax1', 'Varimax2']], IVS6, NOMES6, rep6_ort)
print('Pesos — solução Varimax, 6 componentes, pesos empíricos:')
print(pesos.round(4).to_string())
print(f"\nsoma dos pesos = {pesos['peso'].sum():.6f}")   # tem de dar 1: é uma média ponderada
print(f"dimensão 1 = {100*pesos.loc[pesos['dimensao']==1,'peso'].sum():.1f}%  ·  "
      f"dimensão 2 = {100*pesos.loc[pesos['dimensao']==2,'peso'].sum():.1f}%")   # os 65/35
pesos.rename_axis('variavel').round(6).to_csv(
    CAMINHO_FAT / 'nb04_pesos.csv', sep=';', encoding='utf-8-sig')

Pesos — solução Varimax, 6 componentes, pesos empíricos:
                                  coluna  dimensao   carga    peso
Água inadequada            pct_agua_inad         2  0.9026  0.2233
Esgoto inadequado        pct_esgoto_inad         2  0.6794  0.1266
Razão de moradores       razao_moradores         1  0.5320  0.0716
Analfabetismo 15+            pct_analfab         1  0.8683  0.1909
Renda (invertida)              renda_inv         1  0.9151  0.2120
Cor/raça PPI        pct_raca_pretpardind         1  0.8328  0.1756

soma dos pesos = 1.000000
dimensão 1 = 65.0%  ·  dimensão 2 = 35.0%


In [14]:
# ── Índice 0-1 e escore refinado, e a concordância entre os dois ─────────────
def minmax(df_, colunas):
    """Min-max global, provisório: a normalização por município é do Notebook 03."""
    sub = df_[colunas]
    # (x - min) / (max - min) põe cada variável na escala 0–1, para poderem ser somadas
    return (sub - sub.min()) / (sub.max() - sub.min())

base6 = df.dropna(subset=IVS6).copy()                 # os 87.545 completos, com CD_SETOR
Z01 = minmax(base6, IVS6)                             # as 6 variáveis, cada uma de 0 a 1
# O ÍNDICE: média ponderada. `vetor` casa peso e coluna por NOME, nunca por posição.
base6['indice_01'] = (Z01 * vetor(pesos, IVS6)).sum(axis=1)

# Escore refinado: B = R^-1 A (livro, p. 25).
#
# ATENÇÃO À ESCALA. O modelo foi estimado sobre a matriz de SPEARMAN — ou seja, sobre os
# POSTOS das variáveis, não sobre elas. Os coeficientes B, portanto, se aplicam aos postos
# padronizados. Aplicá-los aos valores brutos padronizados é erro de categoria, e o
# sintoma é imediato e verificável: a variância dos escores deixa de ser 1. As duas
# versões estão calculadas abaixo, e a diferença entre elas é a medida do que a limitação
# 2 do bloco 10 custa.
R6 = a6s['R'].to_numpy()                              # a matriz de Spearman das 6
B = escores_regressao(R6, rot6[['Varimax1', 'Varimax2']].to_numpy())   # B = R⁻¹A

postos6 = pd.DataFrame(postos(base6[IVS6].to_numpy()), index=base6.index, columns=IVS6)
Zr = (postos6 - postos6.mean()) / postos6.std(ddof=0)        # postos padronizados (coerente)
Zb = (base6[IVS6] - base6[IVS6].mean()) / base6[IVS6].std(ddof=0)   # brutos (incoerente)
# ddof=0 divide por n, não por n-1: é o que faz Zᵀ·Z/n dar exatamente a matriz de correlação

escores = Zr.to_numpy() @ B                           # @ é multiplicação de matrizes
escores_brutos = Zb.to_numpy() @ B                    # a mesma conta, na escala errada
base6['escore_F1'], base6['escore_F2'] = escores[:, 0], escores[:, 1]
# O escore composto usa a mesma repartição entre dimensões que o índice 0-1.
base6['escore_composto'] = escores @ rep6_ort         # 65% do F1 + 35% do F2
base6['escore_composto_bruto'] = escores_brutos @ rep6_ort

print(f'variância dos escores sobre POSTOS  : F1 = {escores[:,0].var():.4f} · '
      f'F2 = {escores[:,1].var():.4f}   <- tem de dar 1')
print(f'variância dos escores sobre BRUTOS  : F1 = {escores_brutos[:,0].var():.4f} · '
      f'F2 = {escores_brutos[:,1].var():.4f}')
# Esta trava é a prova de que B = R⁻¹A está certo: com cargas de ACP, a variância é 1.
assert np.allclose([escores[:,0].var(), escores[:,1].var()], 1.0, atol=1e-6), \
    'escores sobre postos deveriam ter variância 1 — B ou a padronização estão errados'
print(f"índice 0-1: min {base6['indice_01'].min():.4f} · mediana "
      f"{base6['indice_01'].median():.4f} · máx {base6['indice_01'].max():.4f}")

# A CONCORDÂNCIA: Spearman compara a ORDEM dos setores, que é o que importa para faixas.
rho = base6[['indice_01', 'escore_composto']].corr(method='spearman').iloc[0, 1]
rho_bruto = base6[['indice_01', 'escore_composto_bruto']].corr(method='spearman').iloc[0, 1]
print(f'\nSpearman índice 0-1 × escore refinado (postos): {rho:.4f}')
print(f'Spearman índice 0-1 × escore refinado (brutos): {rho_bruto:.4f}')
print('critério do plano: acima de 0,95 valida o índice 0-1 apesar de ser "não refinado"')
# Deu 0,924 e 0,945 — os dois abaixo do corte. A causa é de ESCALA, não de instabilidade.
print('VALIDADO' if rho > 0.95 else 'ABAIXO DO CRITÉRIO — ver a leitura no bloco 10')

base6[['CD_SETOR', 'NM_MUN', 'indice_01', 'escore_F1', 'escore_F2',     # uma linha por setor
       'escore_composto', 'escore_composto_bruto']] \
  .round(6).to_csv(CAMINHO_FAT / 'nb04_escores.csv', sep=';', index=False, encoding='utf-8-sig')
print('nb04_escores.csv')

variância dos escores sobre POSTOS  : F1 = 1.0000 · F2 = 1.0000   <- tem de dar 1
variância dos escores sobre BRUTOS  : F1 = 0.8943 · F2 = 0.9696
índice 0-1: min 0.0647 · mediana 0.3540 · máx 0.8303



Spearman índice 0-1 × escore refinado (postos): 0.9241
Spearman índice 0-1 × escore refinado (brutos): 0.9449
critério do plano: acima de 0,95 valida o índice 0-1 apesar de ser "não refinado"
ABAIXO DO CRITÉRIO — ver a leitura no bloco 10


nb04_escores.csv


## 8. Cenários de decisão

Três decisões estão em aberto e são da orientação. O papel deste bloco é **medir o custo de
cada opção**, não escolher.

A métrica não é variância explicada — é **quantos setores mudam de faixa**. Um índice serve
para classificar território; o que importa é se a escolha metodológica muda a
classificação de quem vai receber política pública.

| Cenário | A decisão que ele mede |
|---|---|
| Dois fatores × um fator | Kaiser e Horn retêm um; a teoria do IVS-BH pede dois |
| Pesos empíricos (65/35) × literatura (60/40) | decisão nº 1 da §6.3 do Guia |
| Com × sem o analfabetismo | a política do sigilo: 16.563 setores em jogo |

In [15]:
# ── Os cenários, comparados por mudança de faixa ─────────────────────────────
def faixas(serie):
    """Quatro faixas por quartil — a categorização prevista para o IVS."""
    # rank(method='first') desempata pela ordem de aparição: sem isso, empates fariam o
    # qcut criar faixas de tamanhos diferentes. Assim os quatro quartos ficam iguais.
    return pd.qcut(serie.rank(method='first'), 4, labels=['1 (menor)', '2', '3', '4 (maior)'])

def indice_com_pesos(base_, colunas, vetor_pesos):
    """O índice 0–1 com um vetor de pesos qualquer — é o que permite comparar cenários."""
    Z = minmax(base_, colunas)                      # cada variável em 0–1
    return (Z * np.asarray(vetor_pesos)).sum(axis=1)   # média ponderada, linha a linha

cen = pd.DataFrame(index=base6.index)               # uma coluna por cenário, mesmos setores
cen['dois_fatores'] = base6['indice_01']            # o cenário de REFERÊNCIA

# (a) um fator: os pesos passam a ser o quadrado da carga no primeiro componente.
_, c6 = acp(R6, 1)                                  # extrai UM componente só
p1 = (c6[:, 0] ** 2) / (c6[:, 0] ** 2).sum()        # quadrado da carga, normalizado
cen['um_fator'] = indice_com_pesos(base6, IVS6, p1)

# (b) 60/40 da literatura, mantida a repartição interna de cada dimensão.
pesos_6040 = vetor(pesos, IVS6).copy()              # parte dos pesos empíricos
for j, alvo in ((1, 0.60), (2, 0.40)):              # e só reescala o TOTAL de cada dimensão
    m = (pesos.set_index('coluna').loc[list(IVS6), 'dimensao'] == j).to_numpy()   # máscara
    pesos_6040[m] = alvo * pesos_6040[m] / pesos_6040[m].sum()   # renormaliza para 0,60/0,40
cen['pesos_6040'] = indice_com_pesos(base6, IVS6, pesos_6040)

# (c) sem o analfabetismo: refaz a fatorial nas 5 variáveis restantes e recompõe os pesos.
IVS5 = [c for c in IVS6 if c != 'pct_analfab']
base5 = df.dropna(subset=IVS5).copy()               # sem exigir analfabetismo, sobram mais setores
a5 = adequabilidade(base5[IVS5], [ROTULOS[c] for c in IVS5], 'spearman')
_, c5 = acp(a5['R'].to_numpy(), 2)                  # a fatorial inteira, refeita nas 5
v5 = varimax(c5)
for j in range(2):                                  # mesma convenção de sinal de sempre
    if v5[:, j].sum() < 0:
        v5[:, j] *= -1
ss5 = (v5 ** 2).sum(axis=0); rep5 = ss5 / ss5.sum()   # a repartição própria deste cenário
pesos5 = montar_pesos(pd.DataFrame(v5, index=[ROTULOS[c] for c in IVS5]),
                      IVS5, [ROTULOS[c] for c in IVS5], rep5)
print(f'cenário sem analfabetismo: a fatorial roda em {len(base5):,} setores '
      f'(+{len(base5)-len(base6):,} em relação aos {len(base6):,} completos)')
# Os PESOS vêm da fatorial nos 104.093 — é o que a política do sigilo produziria. Mas o
# ÍNDICE é calculado sobre os mesmos 87.545 da referência, e não sobre os 104.093: min-max
# num suporte maior muda a escala de cada variável, e a comparação passaria a medir duas
# coisas ao mesmo tempo. Aqui o que se mede é só o efeito de tirar o analfabetismo.
cen['sem_analfab'] = indice_com_pesos(base6, IVS5, vetor(pesos5, IVS5))

# Tabelas de contingência entre as faixas, sempre contra o cenário de referência.
ref = faixas(cen['dois_fatores'])                   # as faixas da referência
linhas = []
for nome in [c for c in cen.columns if c != 'dois_fatores']:   # cada cenário alternativo
    alt = faixas(cen[nome])                         # as faixas do cenário
    tab = pd.crosstab(ref, alt)                     # quem foi de qual faixa para qual
    # A MÉTRICA QUE IMPORTA: quantos setores trocaram de faixa. Não é variância explicada —
    # é se a escolha metodológica muda a classificação de quem recebe política pública.
    mudou = int((ref.astype(str) != alt.astype(str)).sum())
    rho_c = cen[['dois_fatores', nome]].corr(method='spearman').iloc[0, 1]   # e a ordem geral
    linhas.append({'cenario': nome, 'setores_que_mudam_de_faixa': mudou,
                   'pct': 100 * mudou / len(cen), 'spearman_com_referencia': rho_c})
    print(f'\n── {nome} × referência (dois fatores, pesos empíricos) ──')
    print(tab.to_string())                          # a diagonal é quem NÃO mudou
    print(f'mudam de faixa: {mudou:,} setores ({100*mudou/len(cen):.1f}%) · '
          f'Spearman {rho_c:.4f}')
    tab.to_csv(CAMINHO_FAT / f'nb04_contingencia_{nome}.csv', sep=';', encoding='utf-8-sig')

resumo_cen = pd.DataFrame(linhas)                   # as três linhas do slide do custo
resumo_cen.round(4).to_csv(CAMINHO_FAT / 'nb04_cenarios.csv', sep=';', index=False, encoding='utf-8-sig')
print('\n' + resumo_cen.round(4).to_string(index=False))

cenário sem analfabetismo: a fatorial roda em 104,093 setores (+16,548 em relação aos 87,545 completos)



── um_fator × referência (dois fatores, pesos empíricos) ──
um_fator      1 (menor)      2      3  4 (maior)
dois_fatores                                    
1 (menor)         21479    408      0          0
2                   336  20325   1225          0
3                    49    897  18560       2380
4 (maior)            23    256   2101      19506
mudam de faixa: 7,675 setores (8.8%) · Spearman 0.9830

── pesos_6040 × referência (dois fatores, pesos empíricos) ──
pesos_6040    1 (menor)      2      3  4 (maior)
dois_fatores                                    
1 (menor)         21783    104      0          0
2                   104  21421    361          0
3                     0    361  20898        627
4 (maior)             0      0    627      21259
mudam de faixa: 2,184 setores (2.5%) · Spearman 0.9993



── sem_analfab × referência (dois fatores, pesos empíricos) ──
sem_analfab   1 (menor)      2      3  4 (maior)
dois_fatores                                    
1 (menor)         21357    530      0          0
2                   527  19999   1360          0
3                     3   1349  19121       1413
4 (maior)             0      8   1405      20473
mudam de faixa: 6,595 setores (7.5%) · Spearman 0.9948

    cenario  setores_que_mudam_de_faixa    pct  spearman_com_referencia
   um_fator                        7675 8.7669                   0.9830
 pesos_6040                        2184 2.4947                   0.9993
sem_analfab                        6595 7.5333                   0.9948


## 8b. A renda sem o valor extremo

A 2ª rodada da EDA recalculou tudo com `renda_media_sem_extremo` — a renda sem o setor
`310620005650366`, de Belo Horizonte. A análise fatorial é **anterior** a essa coluna: os
CSVs de referência de agosto e tudo que veio deles usam `renda_media`.

A pergunta é direta: trocar uma coluna pela outra muda a estrutura, os pesos ou a
classificação? Um único setor em 87.545 não deveria mover nada — mas a renda é a variável
de maior carga do índice, e o extremo em questão foi grande o bastante para justificar uma
rodada inteira da EDA. Medir custa uma célula.

A comparação é feita com tudo o mais igual: mesmas seis variáveis, mesmo recorte, mesma
correlação de Spearman, mesma rotação, mesmos setores no cálculo do índice.

In [16]:
# ── A mesma fatorial, trocando só a coluna de renda ──────────────────────────
# Tudo o mais IGUAL: mesmas 6 variáveis, mesmo recorte, Spearman, Varimax. Só a renda muda.
IVS6_SE = [c if c != 'renda_inv' else 'renda_sem_extremo_inv' for c in IVS6]
NOMES6_SE = NOMES6                        # os rótulos de leitura não mudam
base6_se = df.dropna(subset=IVS6_SE)      # um setor a menos: o próprio extremo

a6_se = adequabilidade(base6_se[IVS6_SE], NOMES6_SE, 'spearman')   # KMO, MSA, Bartlett
_, c6_se = acp(a6_se['R'].to_numpy(), 2)  # a extração
v6_se = varimax(c6_se)                    # e a rotação
for j in range(2):                        # a convenção de sinal, de novo
    if v6_se[:, j].sum() < 0:
        v6_se[:, j] *= -1
ss_se = (v6_se ** 2).sum(axis=0)          # soma dos quadrados por fator
rep_se = ss_se / ss_se.sum()              # a repartição entre as dimensões
pesos_se = montar_pesos(pd.DataFrame(v6_se, index=NOMES6_SE), IVS6_SE, NOMES6_SE, rep_se)

print(f'setores completos: {len(base6_se):,} com a renda sem extremo, '
      f'{len(completo6):,} com a renda completa')
# A tabela do confronto: cada medida, nas duas colunas de renda, e a diferença.
comparacao = pd.DataFrame({
    'com renda_media': [a6s['kmo'], a6s['msa'].min(), 100*rep6_ort[0], 100*rep6_ort[1]],
    'com renda_media_sem_extremo': [a6_se['kmo'], a6_se['msa'].min(),
                                    100*rep_se[0], 100*rep_se[1]],
}, index=['KMO', 'MSA mínimo', 'peso socioeconômico (%)', 'peso saneamento (%)'])
comparacao['diferença'] = comparacao.iloc[:, 1] - comparacao.iloc[:, 0]   # a coluna que decide
print('\n' + comparacao.round(4).to_string())

# E as cargas, uma a uma: é o teste mais duro, porque carga é o resultado da fatorial.
cargas_lado = pd.DataFrame({
    'Varimax1 (renda_media)': rot6['Varimax1'].to_numpy(),
    'Varimax1 (sem extremo)': v6_se[:, 0],
    'dif_1': np.abs(rot6['Varimax1'].to_numpy() - v6_se[:, 0]),
    'Varimax2 (renda_media)': rot6['Varimax2'].to_numpy(),
    'Varimax2 (sem extremo)': v6_se[:, 1],
    'dif_2': np.abs(rot6['Varimax2'].to_numpy() - v6_se[:, 1]),
}, index=NOMES6)
print('\n' + cargas_lado.round(4).to_string())
# Deu 0,0000: as cargas batem até a quarta casa decimal. A troca não muda a estrutura.
print(f"\nmaior diferença absoluta entre cargas: "
      f"{cargas_lado[['dif_1', 'dif_2']].to_numpy().max():.4f}")

setores completos: 87,544 com a renda sem extremo, 87,545 com a renda completa

                         com renda_media  com renda_media_sem_extremo  diferença
KMO                               0.7870                       0.7870    -0.0000
MSA mínimo                        0.7153                       0.7153    -0.0001
peso socioeconômico (%)          65.0106                      65.0110     0.0004
peso saneamento (%)              34.9894                      34.9890    -0.0004

                    Varimax1 (renda_media)  Varimax1 (sem extremo)  dif_1  Varimax2 (renda_media)  Varimax2 (sem extremo)  dif_2
Água inadequada                     0.0864                  0.0864    0.0                  0.9026                  0.9026    0.0
Esgoto inadequado                   0.3917                  0.3917    0.0                  0.6794                  0.6794    0.0
Razão de moradores                  0.5320                  0.5320    0.0                  0.3121                  0.3121    0.

In [17]:
# ── E o que isso faz com a classificação dos setores ─────────────────────────
# Mesmos setores da referência, para que a comparação meça a troca de coluna e mais nada.
# O setor do extremo tem renda_media e NÃO tem renda_media_sem_extremo — a comparação
# roda nos setores que existem nas duas colunas, senão um NaN entraria no quartil.
comum = base6.dropna(subset=IVS6_SE)      # 87.544: um a menos que a referência
print(f'setores comparáveis nas duas colunas: {len(comum):,} '
      f'(de {len(base6):,}; a diferença é o próprio setor do extremo)')
idx_se = indice_com_pesos(comum, IVS6_SE, vetor(pesos_se, IVS6_SE))   # o índice alternativo
faixa_ref, faixa_se = faixas(comum['indice_01']), faixas(idx_se)      # as duas classificações
mudou_se = int((faixa_ref.astype(str) != faixa_se.astype(str)).sum()) # quantos trocam de faixa
rho_se = pd.concat([comum['indice_01'], idx_se], axis=1).corr(method='spearman').iloc[0, 1]

print('\n' + pd.crosstab(faixa_ref, faixa_se).to_string())   # fora da diagonal = mudou
# 278 setores de 87.544 (0,32%), com Spearman de 0,999976: praticamente inócua para a
# FATORIAL (Spearman é invariante a essa escala). Sob a normalização MUNICIPAL do IVS
# (Notebook 03), o efeito é outro — 774 de 87.544 setores mudam de faixa; ver NB4-11 e a
# sensibilidade por município no 04b.
print(f'\nmudam de faixa: {mudou_se:,} setores ({100*mudou_se/len(comum):.2f}%)')
print(f'Spearman entre os dois ordenamentos: {rho_se:.6f}')

# O efeito na classificação entra na MESMA tabela: número citado em deck ou relatório
# tem de sair de arquivo, e não de saída de célula.
comparacao.loc['setores comparáveis'] = [len(base6), len(comum), len(comum) - len(base6)]
comparacao.loc['setores que mudam de faixa'] = [0, mudou_se, mudou_se]
comparacao.loc['% que muda de faixa'] = [0.0, 100*mudou_se/len(comum), 100*mudou_se/len(comum)]
comparacao.loc['Spearman entre os ordenamentos'] = [1.0, rho_se, rho_se - 1.0]
comparacao.round(6).rename_axis('medida').to_csv(     # o CSV que o deck lê
    CAMINHO_FAT / 'nb04_renda_sem_extremo.csv', sep=';', encoding='utf-8-sig')
cargas_lado.rename_axis('variavel').round(4).to_csv(
    CAMINHO_FAT / 'nb04_renda_sem_extremo_cargas.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_renda_sem_extremo.csv · nb04_renda_sem_extremo_cargas.csv')

setores comparáveis nas duas colunas: 87,544 (de 87,545; a diferença é o próprio setor do extremo)



col_0      1 (menor)      2      3  4 (maior)
indice_01                                    
1 (menor)      21825     61      0          0
2                 61  21774     51          0
3                  0     51  21808         27
4 (maior)          0      0     27      21859

mudam de faixa: 278 setores (0.32%)
Spearman entre os dois ordenamentos: 0.999976

nb04_renda_sem_extremo.csv · nb04_renda_sem_extremo_cargas.csv


## 9. Validação externa — os setores de favela

Esta é a validação mais forte disponível ao projeto, e ela não existe em nenhuma das duas
referências metodológicas: o marcador é **externo ao índice**.

`CD_TIPO = 1` marca os setores de Favela e Comunidade Urbana do Censo 2022, uma
classificação oficial do IBGE, verificada setor a setor contra a lista oficial com 100% de
concordância (§6.2.8 do Guia). Nenhuma das seis variáveis do índice foi usada para
construí-la.

Um IVS bem construído tem de separar esses setores dos demais. Isso é teste de validade de
critério: mede se o CONJUNTO de variáveis separa FCU, não se a estrutura fatorial ou os
pesos 65/35 estão certos. Como linha de base (Fase 0, `linha_de_base_nb04.csv`): a renda
invertida sozinha tem AUC 0,8819 e a média de postos com pesos iguais tem AUC 0,8533 —
as duas acima do índice oficial. A AUC confirma o conteúdo socioeconômico das variáveis,
sobretudo a renda; não valida o esquema de ponderação. A comparação mais completa, com
mais linhas de base, está no notebook 04b.

A área sob a curva ROC é calculada pela estatística de postos de Mann–Whitney, que trata
empates corretamente e dispensa dependência nova.

In [18]:
# ── Separação dos setores de FCU ─────────────────────────────────────────────
# CD_TIPO = 1 marca Favela e Comunidade Urbana. to_numeric com errors='coerce' aceita a
# coluna vindo como texto ou como número, sem quebrar se vier algo inesperado.
base6['fcu'] = pd.to_numeric(base6['CD_TIPO'], errors='coerce').eq(1)
n_fcu, n_out = int(base6['fcu'].sum()), int((~base6['fcu']).sum())   # ~ inverte o booleano
print(f'setores de FCU no conjunto completo: {n_fcu:,} · demais: {n_out:,}')

def auc_postos(escore, positivo):
    """AUC pela estatística de Mann–Whitney: (soma dos postos dos positivos - n1(n1+1)/2) / (n1*n0)."""
    r = pd.Series(escore).rank()                    # postos com média nos empates
    n1 = int(positivo.sum()); n0 = len(escore) - n1  # n1 = favelas · n0 = os demais
    # esta fórmula trata empates corretamente e dispensa dependência nova (nada de sklearn)
    return float((r[positivo.to_numpy()].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def curva_roc(escore, positivo, pontos=400):
    """TPR e FPR ao longo de cortes igualmente espaçados nos quantis do escore."""
    # cortes nos quantis, e não no intervalo: assim os pontos da curva ficam bem
    # distribuídos mesmo com o índice concentrado numa faixa estreita. [::-1] inverte,
    # para a curva ser traçada do corte mais alto para o mais baixo.
    cortes = np.quantile(escore, np.linspace(0, 1, pontos))[::-1]
    pos, neg = positivo.to_numpy(), ~positivo.to_numpy()
    tpr = np.array([(escore[pos] >= c).mean() for c in cortes])   # sensibilidade
    fpr = np.array([(escore[neg] >= c).mean() for c in cortes])   # 1 − especificidade
    return np.r_[0, fpr, 1], np.r_[0, tpr, 1]       # np.r_ fecha a curva em (0,0) e (1,1)

val = []
for nome, col in (('índice 0-1', 'indice_01'), ('escore refinado', 'escore_composto')):
    s = base6[col].to_numpy()
    a = auc_postos(s, base6['fcu'])                 # a área sob a curva
    md_fcu = float(base6.loc[base6['fcu'], col].median())      # mediana nos setores de favela
    md_out = float(base6.loc[~base6['fcu'], col].median())     # e nos demais
    val.append({'medida': nome, 'auc': a, 'mediana_fcu': md_fcu,
                'mediana_demais': md_out, 'diferenca': md_fcu - md_out})
    print(f'{nome}: AUC = {a:.4f} · mediana FCU {md_fcu:.4f} × demais {md_out:.4f}')

# LEITURA: a AUC é a probabilidade de que, sorteando um setor de favela e um comum, o de
# favela tenha índice maior. 0,5 é o acaso; 1 é a separação perfeita. Deu 0,813 e 0,862.
validacao = pd.DataFrame(val)
print('\ncritério do plano: AUC acima de 0,75 é separação nítida; abaixo de 0,65 é problema sério')
validacao.round(4).to_csv(CAMINHO_FAT / 'nb04_validacao_fcu.csv', sep=';', index=False, encoding='utf-8-sig')
print(validacao.round(4).to_string(index=False))

setores de FCU no conjunto completo: 18,901 · demais: 68,644
índice 0-1: AUC = 0.8131 · mediana FCU 0.4045 × demais 0.3383
escore refinado: AUC = 0.8616 · mediana FCU 0.7289 × demais -0.2522

critério do plano: AUC acima de 0,75 é separação nítida; abaixo de 0,65 é problema sério
         medida    auc  mediana_fcu  mediana_demais  diferenca
     índice 0-1 0.8131       0.4045          0.3383     0.0662
escore refinado 0.8616       0.7289         -0.2522     0.9811


In [19]:
# ── Figura: curva ROC ────────────────────────────────────────────────────────
# FORMA: uma série é o ponto (o índice), a diagonal é só referência — logo *ênfase*, e não
# duas séries categóricas. A curva vai em clay, cheia e grossa; o acaso em cinza,
# tracejado e fino. A AUC entra como número-herói dentro do gráfico, que é a informação
# que o leitor de fato quer, e dispensa ler a curva.
fpr, tpr = curva_roc(base6['indice_01'].to_numpy(), base6['fcu'])   # os pontos da curva
auc = validacao.loc[0, 'auc']                  # a área, já calculada na célula anterior

fig, ax = plt.subplots(figsize=(4.8, 4.8), facecolor=SURF)   # quadrada: ROC pede 1:1
ax.plot([0, 1], [0, 1], color=CINZA, lw=1.2, ls='--', zorder=1)   # a diagonal = o acaso
ax.plot(fpr, tpr, color=CLAY, lw=2.4, zorder=3)                   # a curva do índice
ax.fill_between(fpr, tpr, alpha=0.08, color=CLAY, zorder=2)       # a área sob a curva = a AUC
ax.annotate('acaso', (0.62, 0.58), color=CINZA, fontsize=9, rotation=39)   # rótulo na diagonal
# o número-herói: quem só olhar a figura já leva a informação principal
ax.text(0.97, 0.10, f'AUC {auc:.3f}'.replace('.', ','), ha='right', color=CLAY, fontsize=20)
ax.text(0.97, 0.045, f'{n_fcu:,} setores de FCU contra {n_out:,}'.replace(',', '.'),
        ha='right', color=CINZA, fontsize=8.5)   # o n de cada grupo, para dimensionar
ax.set_xlabel('falso-positivos (1 − especificidade)', color=CINZA, fontsize=9)
ax.set_ylabel('verdadeiro-positivos (sensibilidade)', color=CINZA, fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)           # ROC vive em [0,1]²: fixar evita distorção
ax.tick_params(colors=CINZA, labelsize=9)
ax.xaxis.set_major_formatter(VIRGULA); ax.yaxis.set_major_formatter(VIRGULA)   # 0,2 e não 0.2
ax.grid(color=CINZA, alpha=0.15, lw=0.8); ax.set_axisbelow(True)   # grade recessiva, atrás
for lado in ('top', 'right'):
    ax.spines[lado].set_visible(False)         # tira as bordas de cima e da direita
for lado in ('left', 'bottom'):
    ax.spines[lado].set_color(CINZA)           # e deixa as outras duas discretas
ax.set_facecolor(SURF)
ax.set_title('O índice separa os setores de favela?', color=TINTA, fontsize=11, loc='left')
salvar(fig, 'nb04_roc_fcu.png')

figura: banco_de_dados/eda/fatorial/figuras/nb04_roc_fcu.png


## 10. Síntese, decisões e limitações

In [20]:
# ── Tabela final e o que fica decidido ───────────────────────────────────────
print('PESOS — solução Varimax, 6 componentes (sem o lixo), pesos empíricos\n')
final = pesos.copy()                       # cópia: o original segue com a dimensão numérica,
                                           # que os cenários do bloco 8 ainda usam
final['peso_pct'] = 100 * final['peso']    # em porcentagem, que é como vai para o deck
final['dimensao'] = final['dimensao'].map({1: 'Socioeconômica', 2: 'Saneamento'})   # nomeia
print(final[['dimensao', 'carga', 'peso_pct']].round(3).to_string())

# O PLACAR DO NOTEBOOK: cada linha é um achado, com o critério ao lado para ser julgado.
print(f"\nrepartição empírica ortogonal : {100*rep6_ort[0]:.1f} / {100*rep6_ort[1]:.1f}")
print(f"repartição empírica oblíqua   : {100*rep6_obl[0]:.1f} / {100*rep6_obl[1]:.1f}")
print(f"referência IVS-BH 2012        : 60,0 / 40,0")     # a literatura, para comparar
print(f"Φ entre os fatores            : {phi6[0,1]:.3f}") # 0,522: a evidência da oblíqua
print(f"\nSpearman índice 0-1 × escore refinado: {rho:.4f}  (critério do plano: > 0,95)")
print(f"AUC contra os setores de FCU        : {auc:.4f}  (critério do plano: > 0,75)")
print(f"renda × cor/raça na matriz fatorada : {par_listwise:.3f}  (limiar do livro: 0,80)")

final.rename_axis('variavel').round(6).to_csv(    # a tabela que o deck e o relatório leem
    CAMINHO_FAT / 'nb04_sintese_pesos.csv', sep=';', encoding='utf-8-sig')
print('\nnb04_sintese_pesos.csv')

PESOS — solução Varimax, 6 componentes (sem o lixo), pesos empíricos

                          dimensao  carga  peso_pct
Água inadequada         Saneamento  0.903    22.334
Esgoto inadequado       Saneamento  0.679    12.655
Razão de moradores  Socioeconômica  0.532     7.165
Analfabetismo 15+   Socioeconômica  0.868    19.087
Renda (invertida)   Socioeconômica  0.915    21.201
Cor/raça PPI        Socioeconômica  0.833    17.558

repartição empírica ortogonal : 65.0 / 35.0
repartição empírica oblíqua   : 65.8 / 34.2
referência IVS-BH 2012        : 60,0 / 40,0
Φ entre os fatores            : 0.522

Spearman índice 0-1 × escore refinado: 0.9241  (critério do plano: > 0,95)
AUC contra os setores de FCU        : 0.8131  (critério do plano: > 0,75)
renda × cor/raça na matriz fatorada : 0.784  (limiar do livro: 0,80)

nb04_sintese_pesos.csv


### O IVS é construto reflexivo ou índice formativo?

É a objeção conceitual mais séria em aberto, e não se resolve com mais dados. Vale enunciá-la
com precisão, porque ela decide se a análise fatorial é o instrumento certo.

**Num construto reflexivo, o latente causa os indicadores.** A vulnerabilidade existiria como
propriedade do território e se *manifestaria* em renda baixa, analfabetismo, saneamento
precário. Os indicadores são intercambiáveis: devem correlacionar-se alto, e retirar um não
muda o que o construto significa. **É esse o modelo que a análise fatorial pressupõe** — dele
vêm o KMO, o Bartlett, a comunalidade e a própria ideia de carga fatorial.

**Num índice formativo, os indicadores constituem o índice.** Vulnerabilidade *é* a combinação
de privações; cada indicador é uma faceta definidora. Eles **não precisam** correlacionar-se,
retirar um muda o significado do índice, e os pesos vêm de teoria ou de decisão política — não
da covariância. Bollen & Lennox (1991) é a formulação canônica da distinção; o manual da
OCDE/JRC (Nardo et al., 2008), que é a referência para indicadores compostos, os trata como
formativos por padrão.

**O que muda, concretamente, se o IVS for formativo:**

| Resultado deste notebook | Leitura reflexiva | Leitura formativa |
|---|---|---|
| Lixo com comunalidade 0,052 | Não pertence ao construto: **retirar** | Faceta de saneamento que as outras não cobrem: **manter** |
| Renda × cor/raça a 0,784 | Bloco coeso, evidência do construto | Multicolinearidade: atrapalha separar as contribuições |
| KMO 0,783 e Bartlett | Provam adequabilidade | **Não se aplicam** — não há modelo de fator comum a testar |
| Pesos 65/35 empíricos | Saem da estrutura latente | Teriam de sair de teoria ou de política pública |

A decisão sobre o lixo **se inverte** entre as duas leituras. É o exemplo mais claro de que
isto não é preciosismo terminológico.

**A saída que os dados sugerem, e que precisa de aval da orientação.** A estrutura encontrada
aponta para um híbrido, e ele é defensável: *dentro* de cada dimensão o modelo se comporta
como reflexivo — renda, analfabetismo e cor/raça correlacionam-se de 0,63 a 0,78 e claramente
manifestam uma mesma posição social do território; água e esgoto, a 0,41, manifestam
infraestrutura de saneamento. *Entre* as duas dimensões a composição é formativa: não há razão
para supor um latente único que cause tanto a falta de água quanto o analfabetismo — o que a
correlação Φ de 0,522 entre os fatores, longe de 1, é consistente em sugerir.

Se esse for o modelo, a consequência prática é limpa e já está medida aqui: a análise fatorial
é legítima para obter os pesos **dentro** de cada bloco, e a repartição **entre** blocos — o
65/35 — é decisão formativa, que os dados não têm como arbitrar. Isso explica, e agora
justifica, por que a escolha entre 65/35 e 60/40 custa apenas 2,5% dos setores: ela nunca foi
uma questão empírica.

**O que falta:** literatura própria. Bollen & Lennox (1991), Diamantopoulos & Winklhofer (2001)
e Edwards (2011) são o núcleo da discussão, e nenhum está lido no projeto. O IVS-BH 2012 e o
ISU de Passarelli-Araujo (2023) não declaram a posição deles, e seria útil saber qual é.

### Três resultados que contrariam o que os documentos previam

Registrados aqui porque a diferença entre achado e erro importa, e nos três casos a
verificação apontou para achado.

1. **A concordância entre o índice 0–1 e o escore refinado não atingiu o critério**, e a
   forma como ela falha é mais informativa do que o número. O plano previa Spearman acima
   de 0,97 e fixou 0,95 como validação. Deu **0,924** com o escore calculado sobre os
   postos — que é o coerente, porque a solução fatorial foi estimada sobre a matriz de
   Spearman — e **0,945** com o escore calculado sobre os valores brutos padronizados, que
   é o incoerente. Os dois ficam acima do piso de 0,90 que obrigaria a trocar o índice
   oficial pelo escore refinado, e os dois ficam abaixo de 0,95.

   O sentido da diferença é o ponto: **quanto mais coerente o escore fica com o modelo
   fatorial, mais ele se afasta do índice planejado**. A razão é que o índice 0–1 é
   min-max de valores **brutos** e o modelo fatorial vive nos **postos**. Não é
   instabilidade amostral — o bootstrap mediu a incerteza dos pesos em 0,59 ponto
   percentual, que é desprezível. É a limitação 2 abaixo cobrando o seu preço, e ela não
   se resolve com mais reamostragem: ou o índice passa a ser composto sobre postos, ou a
   fatorial passa a ser estimada sobre Pearson, ou a divergência é declarada. As três são
   defensáveis; escolher sem saber que se está escolhendo, não.
2. **A repartição oblíqua não tem uma única resposta.** O plano previa que a rotação
   oblíqua deslocasse os pesos de 65/35 na direção dos 60/40 do IVS-BH. Pela matriz
   padrão dá 65,8/34,2 (meio ponto no sentido oposto); pela matriz estrutura dá
   59,6/40,4 (mais perto dos 60/40). Nenhuma das duas é partição aditiva com fatores
   correlacionados — não se conclui que a oblíqua "afasta" nem que "aproxima" os pesos
   da literatura; as duas ficam lado a lado no bloco 5.
3. **Nenhum par da matriz fatorada cruza o limiar de multicolinearidade.** Ver a
   limitação 3 abaixo.

Um quarto ponto, que confirma em vez de contrariar: na solução de sete variáveis, o eixo
principal atribui ao lixo comunalidade de **0,052**, contra 0,859 da ACP — a maior
diferença de comunalidade da tabela do bloco 4, e coerente com a SMC do lixo (0,106), a
mais baixa entre as sete. (O 0,836 de `dif_2` na mesma tabela compara colunas de fatores
diferentes — ACP_2 é o fator do lixo, PAF_2 é o de saneamento — e não mede divergência
de método.) O lixo não tem variância comum com o construto: o fator próprio que forma
na ACP é variância específica, que a análise fatorial não conta.

### As decisões que vão para a orientação

Nenhuma delas se fecha aqui. O que este notebook entrega é o custo de cada opção.

| # | Decisão | Evidência produzida |
|---|---|---|
| 1 | Pesos empíricos ou 60/40 da literatura | os dois convergem; a tabela de contingência do bloco 8 diz quantos setores mudam de faixa |
| 2 | Destino do indicador de lixo | forma fator próprio; sem ele a variância acumulada sobe e a estrutura teórica aparece limpa |
| 3 | Política do sigilo no analfabetismo | o cenário sem a variável está no bloco 8, com os setores recuperados e a mudança de faixa |
| 4 | Um fator ou dois | Kaiser e Horn dizem um na solução sem lixo; a teoria diz dois; o custo está no bloco 8 |
| 5 | Rotação ortogonal ou oblíqua | Φ está calculado no bloco 5, e com ele a repartição oblíqua dos pesos |

### Limitações declaradas

1. **O Bartlett é vazio nesta escala.** Com 87.545 casos o teste rejeita H₀ por construção
   (livro, p. 43). A adequabilidade se apoia no KMO e nos MSA.
2. **ACP sobre matriz de Spearman é ACP de postos.** As cargas se referem a posições
   relativas, não a magnitudes, e os escores herdam essa natureza ordinal. A solução de
   Pearson está no bloco 2 como sensibilidade.
3. **Multicolinearidade renda × cor/raça a 0,784** na matriz efetivamente fatorada —
   **abaixo** do limiar de 0,80 da p. 42. O −0,811 que os documentos do projeto citam é a
   correlação par a par da EDA, calculada sobre os 104.108 setores do recorte, não sobre
   os 87.545 da matriz listwise. Os dois números estão certos e medem conjuntos
   diferentes; o artigo precisa citar o da matriz que foi fatorada.
4. **Viés não aleatório do sigilo:** 16.563 setores (15,9%) perdidos por *listwise*, quase
   todos pelo analfabetismo, e o sigilo incide nos setores de melhor situação (§6.2.6 do
   Guia). A amostra é enviesada para os mais vulneráveis. O livro da Enap não trata de
   dados faltantes em nenhuma das 74 páginas.
5. **Dependência espacial não tratada.** A análise fatorial pressupõe unidades
   independentes; setores vizinhos não são. A autocorrelação infla a covariação e, com
   ela, autovalores e cargas. O I de Moran dos escores, na etapa de geoprocessamento, dará
   a medida do problema.
6. **Falácia ecológica.** As unidades são territórios. Toda carga descreve covariação entre
   setores e nada afirma sobre indivíduos.
7. **A padronização usada aqui é min-max global e provisória.** A normalização por
   município é do Notebook 03, e a ordem entre as duas etapas foi medida: fatorar depois de
   normalizar derruba o KMO para 0,720 e a repartição para 56/44.
8. **`renda_media_sem_extremo` já foi testada, no bloco 8b acima.** A fatorial em si não
   muda (Spearman é invariante à escala; ver o bloco 8b). O efeito que fica: sob a
   normalização MUNICIPAL que o IVS vai usar (Notebook 03, não este), a troca de renda
   pesa mais — 774 de 87.544 setores mudam de faixa (NB4-11). Qual renda vai ao índice
   final continua sendo decisão pendente da orientadora.
9. **Reflexivo ou formativo.** Se o IVS é um construto que *causa* os indicadores
   (reflexivo, que é o que a análise fatorial pressupõe) ou um índice *composto por* eles
   (formativo, em que a fatorial não seria o instrumento adequado) é questão conceitual em
   aberto, e precisa de literatura própria e decisão da orientação.

### Reprodutibilidade

Matemática em `src/ivs_censo/fatorial.py`, testada em `tests/test_fatorial.py`. Entrada:
o `.db` da entrega. Saídas em `banco_de_dados/eda/fatorial/` com prefixo `nb04_`, e figuras
em `figuras/`. Rodar com:

```
jupyter execute notebooks/Fase3_EDA_ELSI/04_Analise_Fatorial.ipynb
```